# MFV-GDN — UNSW-NB15

Portable repository version. Dataset files can be placed in `data/` or the repository root.


## Repository Setup

Expected dataset files:
- `data/UNSW_NB15_training-set.csv`
- `data/UNSW_NB15_testing-set.csv`

The same files may also be placed in the repository root. Use `MFV_DATA_DIR` to point to another local dataset directory.


## Block 1: Environment Check


In [ ]:
import sys, os, socket, platform, shutil, subprocess, importlib.util

print("Host:", socket.gethostname())
print("OS:", platform.platform())
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Working directory:", os.getcwd())

required_modules = {
    "numpy": "NumPy",
    "pandas": "Pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "Matplotlib",
    "networkx": "NetworkX",
    "torch": "PyTorch",
    "torch_geometric": "PyTorch Geometric",
    "tensorflow": "TensorFlow",
    "xgboost": "XGBoost",
    "lightgbm": "LightGBM",
    "shap": "SHAP",
    "lime": "LIME",
}

missing = [
    label for mod, label in required_modules.items()
    if importlib.util.find_spec(mod) is None
]

if missing:
    print("Missing packages:", ", ".join(missing))
    print("Install repository dependencies with:")
    print("  pip install -r requirements.txt")
else:
    print("Required Python packages: OK")

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    print("\nGPU status:")
    subprocess.run(
        [
            nvidia_smi,
            "--query-gpu=name,driver_version,memory.total,memory.free",
            "--format=csv,noheader"
        ],
        check=False
    )
else:
    print("\nnvidia-smi not found. CPU execution is still supported.")


## Block 2: Imports and Settings


In [ ]:
import os, random, time, math, hashlib, warnings, itertools
from pathlib import Path
from dataclasses import dataclass
from collections import Counter

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")

# Repository paths
PROJECT_ROOT = Path(os.environ.get("MFV_PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("MFV_DATA_DIR", PROJECT_ROOT / "data")).resolve()

def resolve_dataset_file(filename):
    candidates = [
        DATA_DIR / filename,
        PROJECT_ROOT / filename,
    ]
    for path in candidates:
        if path.is_file():
            return path
    return candidates[0]

TRAIN_CSV = resolve_dataset_file("UNSW_NB15_training-set.csv")
TEST_CSV = resolve_dataset_file("UNSW_NB15_testing-set.csv")

OUTPUT_DIR = Path(
    os.environ.get("MFV_OUTPUT_DIR", PROJECT_ROOT / "outputs")
).resolve()
FIGURE_DIR = OUTPUT_DIR / "figures"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
# Headless plotting
if not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")

import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)

# Save figures automatically in headless sessions.
if not os.environ.get("DISPLAY"):
    _figure_counter = itertools.count(1)
    def _talon_show(*args, **kwargs):
        fig_nums = list(plt.get_fignums())
        for fig_num in fig_nums:
            fig = plt.figure(fig_num)
            idx = next(_figure_counter)
            out_file = FIGURE_DIR / f"figure_{idx:03d}.png"
            fig.savefig(out_file, dpi=300, bbox_inches="tight")
            print(f"Saved figure: {out_file}")
        plt.close("all")
    plt.show = _talon_show

from sklearn.preprocessing import StandardScaler, OrdinalEncoder, label_binarize
from sklearn.ensemble import (
    RandomForestClassifier, StackingClassifier, BaggingClassifier,
    AdaBoostClassifier, GradientBoostingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedKFold, StratifiedGroupKFold, train_test_split
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix,
    roc_curve, precision_recall_curve, auc
)
from sklearn.neighbors import BallTree, NearestNeighbors
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data as PyGData
from torch_geometric.nn import GCNConv, GATConv

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, Conv1D, BatchNormalization,
    Bidirectional, LSTM, MultiHeadAttention,
    GlobalAveragePooling1D, LayerNormalization, Add
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
tf.random.set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CPU_THREADS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))
torch.set_num_threads(max(1, CPU_THREADS))
os.environ.setdefault("OMP_NUM_THREADS", str(max(1, CPU_THREADS)))
os.environ.setdefault("MKL_NUM_THREADS", str(max(1, CPU_THREADS)))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(gpu, True)
except Exception:
    pass

CLASS_NAMES = [
    "Normal", "Fuzzers", "Analysis", "Backdoor", "DoS",
    "Exploits", "Generic", "Reconnaissance", "Shellcode", "Worms"
]
NUM_CLASSES = len(CLASS_NAMES)

OUTER_FOLDS = 5
INNER_CF_FOLDS = 5

HHO_ALPHA = 0.95
HHO_HAWKS = 20
HHO_ITERS = 25
HHO_RF_TREES = 100

NOISE_DIM = 64
WGAN_EPOCHS = 300
WGAN_LR = 2e-4
WGAN_GP_LAMBDA = 10.0
WGAN_BATCH = 128
WGAN_N_CRITIC = 5
WGAN_NOVELTY_MIN_DIST = 1e-6

GNN_HIDDEN = 64
GNN_OUT = 32
GNN_HEADS = 4
GNN_DROPOUT = 0.30
GNN_EPOCHS = 60
GNN_PRETRAIN_EPOCHS = 60
GNN_LR = 1e-3
GNN_EDGE_BUDGET = 4000

NSA_SELF_MAX = 8000
NSA_RHO = 95.0
NSA_N_CAND = 120000
NSA_N_DET = 600
NSA_R_MIN = 0.15
NSA_R_MAX = 5.0
NSA_BETA = 0.30
NSA_DELTA = 1e-6
NSA_GAMMA = 6.0
NSA_W_DIST = 0.60
NSA_W_DET = 0.40
NSA_EPS = 1e-9
NSA_K = 10

CSA_N_MEM = 15
CSA_N_CLONES = 8
CSA_N_GEN = 12
CSA_ETA = 0.25

CNN_CONV_FILTERS = (64, 32)
CNN_BILSTM_UNITS = 64
CNN_MHA_HEADS = 4
CNN_MHA_KEY_DIM = 16
CNN_DENSE_UNITS = (128, 64)
CNN_DROPOUTS = (0.25, 0.35, 0.25)
CNN_OPTIMIZER = "Adam"

CNN_EPOCHS = 80
CNN_BATCH = 128
CNN_LR = 1e-3

RUN_NESTED_CV = os.environ.get("MFV_RUN_NESTED_CV", "1") == "1"
RUN_FINAL_LOCKED_TEST = os.environ.get("MFV_RUN_FINAL_TEST", "1") == "1"
RUN_COMPARATIVE_MODELS = os.environ.get("MFV_RUN_COMPARATIVE_MODELS", "1") == "1"
RUN_COMPARATIVE_CV = os.environ.get("MFV_RUN_COMPARATIVE_CV", "1") == "1"
RUN_EXPLAINABILITY = os.environ.get("MFV_RUN_EXPLAINABILITY", "1") == "1"
COMPARISON_CV_FOLDS = 5

FAST_DEBUG = os.environ.get("MFV_GDN_FAST_DEBUG", "0") == "1"
if FAST_DEBUG:
    HHO_HAWKS, HHO_ITERS = 6, 5
    WGAN_EPOCHS = 5
    GNN_EPOCHS = 5
    GNN_PRETRAIN_EPOCHS = 5
    NSA_N_CAND, NSA_N_DET = 5000, 100
    INNER_CF_FOLDS = 3
    CNN_EPOCHS = 5

print("Dataset path:", DATASET_PATH)
print("Output directory:", OUTPUT_DIR)
print("CPU threads:", CPU_THREADS)
print("Torch device:", DEVICE)
print("TensorFlow GPUs:", tf.config.list_physical_devices("GPU"))
print("Full experiment settings:", not FAST_DEBUG)

import networkx as nx
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap
from lime.lime_tabular import LimeTabularExplainer



## Block 3: Data Loading, Split Control, and Labels


In [ ]:
if not TRAIN_CSV.is_file():
    raise FileNotFoundError(
        f"Training CSV not found: {TRAIN_CSV}\n"
        "Place UNSW_NB15_training-set.csv in data/ or the repository root, "
        "or set MFV_DATA_DIR."
    )

if not TEST_CSV.is_file():
    raise FileNotFoundError(
        f"Testing CSV not found: {TEST_CSV}\n"
        "Place UNSW_NB15_testing-set.csv in data/ or the repository root, "
        "or set MFV_DATA_DIR."
    )

train_path = TRAIN_CSV
test_path = TEST_CSV

train_raw = pd.read_csv(train_path, low_memory=False)
test_raw = pd.read_csv(test_path, low_memory=False)

print("Project root:", PROJECT_ROOT)
print("Train:", train_path, train_raw.shape)
print("Test :", test_path, test_raw.shape)

UNSW_ATTACK_MAP = {
    "normal": 0, "fuzzers": 1, "analysis": 2,
    "backdoor": 3, "backdoors": 3, "dos": 4,
    "exploits": 5, "generic": 6, "reconnaissance": 7,
    "shellcode": 8, "worms": 9
}

TARGET_LIKE = {
    "label", "attack_cat", "label_name", "binary_label", "multi_label"
}
ID_LIKE = {"id", "row_id", "index"}

def engineer_labels(df):
    out = df.copy()
    if "attack_cat" in out.columns:
        label_name = out["attack_cat"].astype(str).str.strip()
    elif "label" in out.columns and not pd.api.types.is_numeric_dtype(out["label"]):
        label_name = out["label"].astype(str).str.strip()
    else:
        raise ValueError("UNSW-NB15 attack category column 'attack_cat' was not found.")

    out["label_name"] = label_name
    out["multi_label"] = label_name.str.lower().map(UNSW_ATTACK_MAP)
    if out["multi_label"].isna().any():
        bad = sorted(out.loc[out["multi_label"].isna(), "label_name"].unique())
        raise ValueError(f"Unknown attack categories: {bad}")
    out["multi_label"] = out["multi_label"].astype(int)

    if "label" in out.columns and pd.api.types.is_numeric_dtype(out["label"]):
        out["binary_label"] = out["label"].astype(int)
    else:
        out["binary_label"] = (out["multi_label"] != 0).astype(int)
    return out

def raw_feature_columns(df):
    return [
        c for c in df.columns
        if c not in TARGET_LIKE
        and c.lower() not in ID_LIKE
        and not c.startswith("_")
    ]

NEAR_DUP_TOLERANCES = {}

def signature_frame(df, near=False):
    cols = raw_feature_columns(df)
    work = pd.DataFrame(index=df.index)

    for c in cols:
        s = df[c]
        if pd.api.types.is_numeric_dtype(s):
            vals = pd.to_numeric(s, errors="coerce").astype(float)
            if near:
                if c in NEAR_DUP_TOLERANCES:
                    delta = float(NEAR_DUP_TOLERANCES[c])
                else:
                    non_na = vals.dropna()
                    is_int_like = (
                        len(non_na) > 0 and
                        np.allclose(non_na.values, np.round(non_na.values), atol=1e-12)
                    )
                    delta = 1.0 if is_int_like else 1e-6
                q = np.round(vals / max(delta, 1e-12))
                work[c] = q.map(lambda x: "<NA>" if pd.isna(x) else str(int(x)))
            else:
                work[c] = vals.map(lambda x: "<NA>" if pd.isna(x) else f"{x:.12g}")
        else:
            work[c] = s.astype(str).str.strip().str.lower().fillna("<NA>")

    serialized = work.astype(str).agg("\x1f".join, axis=1)
    return serialized.map(lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest())

def split_first_duplicate_audit(train_df, test_df):
    tr = train_df.copy()
    te = test_df.copy()

    tr["_exact_sig"] = signature_frame(tr, near=False)
    te["_exact_sig"] = signature_frame(te, near=False)
    tr["_near_sig"]  = signature_frame(tr, near=True)
    te["_near_sig"]  = signature_frame(te, near=True)

    exact_test = set(te["_exact_sig"])
    near_test  = set(te["_near_sig"])

    collision = tr["_exact_sig"].isin(exact_test) | tr["_near_sig"].isin(near_test)
    n_removed = int(collision.sum())
    tr = tr.loc[~collision].copy()

    tr["_dup_group"] = tr["_near_sig"]
    te["_dup_group"] = te["_near_sig"]

    print(f"Cross-partition duplicate/near-duplicate training rows removed: {n_removed}")
    return tr.reset_index(drop=True), te.reset_index(drop=True)

train_raw_audited, test_raw_locked = split_first_duplicate_audit(train_raw, test_raw)

dev_df  = engineer_labels(train_raw_audited)
test_df = engineer_labels(test_raw_locked)

print("\nDevelopment binary distribution:")
print(dev_df["binary_label"].value_counts().sort_index())
print("\nDevelopment multi-class distribution:")
print(dev_df["multi_label"].value_counts().sort_index().rename(index=dict(enumerate(CLASS_NAMES))))
print("\nLocked test multi-class distribution:")
print(test_df["multi_label"].value_counts().sort_index().rename(index=dict(enumerate(CLASS_NAMES))))


## Block 4: Preprocessing


In [ ]:
class TrainingPreprocessor:
    def __init__(self):
        self.feature_cols = None
        self.num_cols = None
        self.cat_cols = None
        self.num_medians = None
        self.cat_fill = "__MISSING__"
        self.scaler = None
        self.encoder = None
        self.feature_names = None

    def fit(self, df):
        self.feature_cols = raw_feature_columns(df)
        self.num_cols = [c for c in self.feature_cols if pd.api.types.is_numeric_dtype(df[c])]
        self.cat_cols = [c for c in self.feature_cols if c not in self.num_cols]

        self.num_medians = {
            c: pd.to_numeric(df[c], errors="coerce").median()
            for c in self.num_cols
        }

        X_num = pd.DataFrame({
            c: pd.to_numeric(df[c], errors="coerce").fillna(self.num_medians[c])
            for c in self.num_cols
        })

        self.scaler = StandardScaler()
        if self.num_cols:
            self.scaler.fit(X_num[self.num_cols])

        if self.cat_cols:
            X_cat = df[self.cat_cols].astype("object").where(
                df[self.cat_cols].notna(), self.cat_fill
            ).astype(str)
            self.encoder = OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
            self.encoder.fit(X_cat)

        self.feature_names = self.num_cols + self.cat_cols
        return self

    def transform(self, df):
        work = pd.DataFrame(index=df.index)
        for c in self.feature_cols:
            work[c] = df[c] if c in df.columns else np.nan

        parts = []
        if self.num_cols:
            X_num = pd.DataFrame({
                c: pd.to_numeric(work[c], errors="coerce").fillna(self.num_medians[c])
                for c in self.num_cols
            })
            parts.append(self.scaler.transform(X_num[self.num_cols]))

        if self.cat_cols:
            X_cat = work[self.cat_cols].astype("object").where(
                work[self.cat_cols].notna(), self.cat_fill
            ).astype(str)
            parts.append(self.encoder.transform(X_cat))

        return np.hstack(parts).astype(np.float32)

def group_stratified_split_indices(df, y_col="multi_label", n_splits=5, fold=0):
    y = df[y_col].to_numpy()
    groups = df["_dup_group"].astype(str).to_numpy()
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    splits = list(splitter.split(np.zeros(len(df)), y, groups))
    return splits[fold % len(splits)]

def raw_graph_meta(df):
    proto_col = "proto" if "proto" in df.columns else ("protocol_type" if "protocol_type" in df.columns else None)
    service_col = "service" if "service" in df.columns else ("state" if "state" in df.columns else None)
    if proto_col is None or service_col is None:
        raise ValueError("Protocol/service fields required for graph construction were not found.")

    return pd.DataFrame({
        "protocol": df[proto_col].astype(str).str.strip().str.lower().fillna("<UNK>"),
        "service":  df[service_col].astype(str).str.strip().str.lower().fillna("<UNK>")
    }).reset_index(drop=True)

print("Preprocessing utilities ready.")


## Block 5: HHO Feature Selection


In [ ]:
def hho_fitness(mask, Xtr, Xva, ytr, yva):
    sel = np.flatnonzero(mask)
    if len(sel) == 0:
        return -np.inf

    clf = RandomForestClassifier(
        n_estimators=HHO_RF_TREES,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    )
    clf.fit(Xtr[:, sel], ytr)
    pred = clf.predict(Xva[:, sel])

    bal = balanced_accuracy_score(yva, pred)
    sparsity = 1.0 - len(sel) / Xtr.shape[1]
    return HHO_ALPHA * bal + (1.0 - HHO_ALPHA) * sparsity

def sigmoid_binary(position, rng):
    sig = 1.0 / (1.0 + np.exp(-np.clip(position, -30, 30)))
    mask = (rng.random(position.shape) < sig).astype(np.int8)
    if mask.sum() == 0:
        mask[rng.integers(0, len(mask))] = 1
    return mask

def harris_hawks_feature_selection(
    Xtr, Xva, ytr, yva,
    n_hawks=HHO_HAWKS, n_iter=HHO_ITERS, seed=SEED
):
    rng = np.random.default_rng(seed)
    d = Xtr.shape[1]
    hawks = rng.normal(0.0, 1.0, size=(n_hawks, d))

    rabbit_pos = hawks[0].copy()
    rabbit_mask = sigmoid_binary(rabbit_pos, rng)
    rabbit_score = -np.inf
    history = []

    for t in range(n_iter):
        for i in range(n_hawks):
            mask = sigmoid_binary(hawks[i], rng)
            score = hho_fitness(mask, Xtr, Xva, ytr, yva)
            if score > rabbit_score:
                rabbit_score = score
                rabbit_pos = hawks[i].copy()
                rabbit_mask = mask.copy()

        E1 = 2.0 * (1.0 - t / max(n_iter, 1))

        for i in range(n_hawks):
            E0 = 2.0 * rng.random() - 1.0
            E = E1 * E0
            q = rng.random()
            J = 2.0 * (1.0 - rng.random())
            Xi = hawks[i].copy()

            if abs(E) >= 1.0:
                Xrand = hawks[rng.integers(0, n_hawks)]
                if q < 0.5:
                    r1, r2 = rng.random(), rng.random()
                    hawks[i] = Xrand - r1 * np.abs(Xrand - 2.0 * r2 * Xi)
                else:
                    Xm = hawks.mean(axis=0)
                    hawks[i] = (rabbit_pos - Xm) - rng.random(d) * (
                        -1.0 + 2.0 * rng.random(d)
                    )
            else:
                r = rng.random()
                if r >= 0.5 and abs(E) >= 0.5:
                    hawks[i] = (rabbit_pos - Xi) - E * np.abs(J * rabbit_pos - Xi)
                elif r >= 0.5 and abs(E) < 0.5:
                    hawks[i] = rabbit_pos - E * np.abs(rabbit_pos - Xi)
                else:
                    if abs(E) >= 0.5:
                        Y = rabbit_pos - E * np.abs(J * rabbit_pos - Xi)
                    else:
                        Y = rabbit_pos - E * np.abs(J * rabbit_pos - hawks.mean(axis=0))
                    Z = Y + rng.normal(0, 0.01, d)
                    y_mask = sigmoid_binary(Y, rng)
                    z_mask = sigmoid_binary(Z, rng)
                    y_score = hho_fitness(y_mask, Xtr, Xva, ytr, yva)
                    z_score = hho_fitness(z_mask, Xtr, Xva, ytr, yva)
                    hawks[i] = Y if y_score >= z_score else Z

        history.append(rabbit_score)
        print(
            f"HHO iteration {t+1:02d}/{n_iter}: "
            f"best={rabbit_score:.6f}, selected={rabbit_mask.sum()}"
        )

    return np.flatnonzero(rabbit_mask), history

def fit_hho_preprocessor(fit_df):
    in_tr_idx, in_va_idx = group_stratified_split_indices(
        fit_df, y_col="multi_label", n_splits=5, fold=0
    )
    in_tr = fit_df.iloc[in_tr_idx].reset_index(drop=True)
    in_va = fit_df.iloc[in_va_idx].reset_index(drop=True)

    pre_inner = TrainingPreprocessor().fit(in_tr)
    X_in_tr = pre_inner.transform(in_tr)
    X_in_va = pre_inner.transform(in_va)

    sel_idx, hist = harris_hawks_feature_selection(
        X_in_tr, X_in_va,
        in_tr["multi_label"].to_numpy(),
        in_va["multi_label"].to_numpy()
    )

    final_pre = TrainingPreprocessor().fit(fit_df)
    if final_pre.feature_names != pre_inner.feature_names:
        raise RuntimeError("Feature schema changed between HHO inner fit and full fit.")

    sel_names = [final_pre.feature_names[i] for i in sel_idx]
    print("HHO selected", len(sel_idx), "features:", sel_names)
    return final_pre, sel_idx, sel_names, hist


## Block 6: WGAN-GP Oversampling


In [ ]:

class WGANGenerator(nn.Module):
    def __init__(self, noise_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.LeakyReLU(0.2),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.2),
            nn.Linear(128, out_dim), nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


class WGANCritic(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x)


def gradient_penalty(critic, real, fake):
    eps = torch.rand(real.size(0), 1, device=real.device)
    xhat = (eps * real + (1.0 - eps) * fake).requires_grad_(True)
    yhat = critic(xhat)
    grad = torch.autograd.grad(
        outputs=yhat,
        inputs=xhat,
        grad_outputs=torch.ones_like(yhat),
        create_graph=True,
        retain_graph=True
    )[0]
    return ((grad.norm(2, dim=1) - 1.0) ** 2).mean()


def train_wgan_gp_one_class(X_class, seed=SEED):
    torch.manual_seed(seed)
    X_class = np.asarray(X_class, dtype=np.float32)

    x_min = X_class.min(axis=0)
    x_max = X_class.max(axis=0)
    span = np.where(x_max - x_min > 1e-12, x_max - x_min, 1.0)

    # Class-specific training bounds.
    X_norm = 2.0 * (X_class - x_min) / span - 1.0
    x_tensor = torch.tensor(X_norm, dtype=torch.float32, device=DEVICE)

    generator = WGANGenerator(NOISE_DIM, X_class.shape[1]).to(DEVICE)
    critic = WGANCritic(X_class.shape[1]).to(DEVICE)

    opt_g = torch.optim.Adam(generator.parameters(), lr=WGAN_LR, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(critic.parameters(), lr=WGAN_LR, betas=(0.5, 0.999))

    batch = min(WGAN_BATCH, len(x_tensor))

    for epoch in range(1, WGAN_EPOCHS + 1):
        for _ in range(WGAN_N_CRITIC):
            idx = torch.randint(0, len(x_tensor), (batch,), device=DEVICE)
            real = x_tensor[idx]

            z = torch.randn(batch, NOISE_DIM, device=DEVICE)
            fake = generator(z).detach()

            gp = gradient_penalty(critic, real, fake)
            loss_d = critic(fake).mean() - critic(real).mean() + WGAN_GP_LAMBDA * gp

            opt_d.zero_grad()
            loss_d.backward()
            opt_d.step()

        z = torch.randn(batch, NOISE_DIM, device=DEVICE)
        fake = generator(z)
        loss_g = -critic(fake).mean()

        opt_g.zero_grad()
        loss_g.backward()
        opt_g.step()

        if epoch == 1 or epoch % 100 == 0 or epoch == WGAN_EPOCHS:
            print(
                f"    epoch {epoch:03d}/{WGAN_EPOCHS}: "
                f"D={loss_d.item():.4f}, G={loss_g.item():.4f}"
            )

    return generator, x_min, x_max, span


def row_hashes(X, decimals=8):
    rounded = np.round(np.asarray(X, dtype=np.float64), decimals)
    return {
        hashlib.sha256(np.ascontiguousarray(row).tobytes()).hexdigest()
        for row in rounded
    }


def enforce_selected_feature_validity(
    synth, real_class_X, selected_feature_names, categorical_raw_names
):
    """
    Synthetic values remain inside the class-specific training range.
    For ordinal-encoded categorical features, each generated value is snapped
    to a category value actually observed in the current training fold.
    """
    synth = np.asarray(synth, dtype=np.float32).copy()

    for j, name in enumerate(selected_feature_names):
        if name in categorical_raw_names:
            allowed = np.unique(real_class_X[:, j])
            if len(allowed):
                nearest = np.abs(synth[:, j, None] - allowed[None, :]).argmin(axis=1)
                synth[:, j] = allowed[nearest]
    return synth


def wgan_gp_oversample(
    X,
    y_mc,
    meta,
    selected_feature_names,
    categorical_raw_names,
    forbidden_arrays=None,
    seed=SEED
):
    rng = np.random.default_rng(seed)
    X = np.asarray(X, dtype=np.float32)
    y_mc = np.asarray(y_mc, dtype=int)
    meta = meta.reset_index(drop=True).copy()
    forbidden_arrays = forbidden_arrays or []

    counts = Counter(y_mc.tolist())
    target = max(counts.values())
    print("WGAN-GP counts before:", dict(sorted(counts.items())))

    X_parts = [X]
    y_parts = [y_mc]
    meta_parts = [meta]
    audit_rows = []

    forbidden_hashes = row_hashes(X)
    for arr in forbidden_arrays:
        if arr is not None and len(arr):
            forbidden_hashes |= row_hashes(arr)

    for cls in sorted(counts):
        need = target - counts[cls]
        if need <= 0:
            continue

        cls_idx = np.flatnonzero(y_mc == cls)
        Xc = X[cls_idx]
        print(f"Class {cls} ({CLASS_NAMES[cls]}): {len(Xc)} real -> +{need}")

        generator, x_min, x_max, span = train_wgan_gp_one_class(
            Xc, seed=seed + int(cls)
        )
        generator.eval()

        accepted = []
        accepted_meta = []
        accepted_nn_distance = []

        nn_real = NearestNeighbors(n_neighbors=1).fit(Xc)

        attempts = 0
        generated_total = 0
        exact_rejected = 0
        novelty_rejected = 0
        novelty_checked = 0

        while sum(len(a) for a in accepted) < need and attempts < 20:
            attempts += 1
            remaining = need - sum(len(a) for a in accepted)
            draw = min(max(remaining * 2, 256), 8192)

            with torch.no_grad():
                z = torch.randn(draw, NOISE_DIM, device=DEVICE)
                generated_norm = generator(z).cpu().numpy()

            synth = ((generated_norm + 1.0) / 2.0) * span + x_min
            synth = np.clip(synth, x_min, x_max)

            # Validate categorical fields.
            synth = enforce_selected_feature_validity(
                synth,
                Xc,
                selected_feature_names,
                categorical_raw_names
            )

            generated_total += len(synth)

            # Exact-duplicate check.
            keep = np.ones(len(synth), dtype=bool)
            hashes = []
            for i, row in enumerate(np.round(synth, 8)):
                h = hashlib.sha256(
                    np.ascontiguousarray(row, dtype=np.float64).tobytes()
                ).hexdigest()
                hashes.append(h)
                if h in forbidden_hashes:
                    keep[i] = False

            exact_rejected += int((~keep).sum())
            synth = synth[keep]
            hashes = [h for h, k in zip(hashes, keep) if k]

            if len(synth) == 0:
                continue

            # Nearest-neighbour novelty check.
            dmin, nn_idx = nn_real.kneighbors(synth, n_neighbors=1)
            dmin = dmin.ravel()
            nn_idx = nn_idx.ravel()

            novelty_checked += len(dmin)
            novel = dmin >= WGAN_NOVELTY_MIN_DIST
            novelty_rejected += int((~novel).sum())

            synth = synth[novel]
            dmin = dmin[novel]
            nn_idx = nn_idx[novel]
            hashes = [h for h, k in zip(hashes, novel) if k]

            if len(synth) == 0:
                continue

            # Copy protocol/service metadata from the nearest real same-class
            # training flow.
            cls_meta = meta.iloc[cls_idx].reset_index(drop=True)
            synth_meta = cls_meta.iloc[nn_idx].reset_index(drop=True)

            take = min(remaining, len(synth))
            accepted.append(synth[:take].astype(np.float32))
            accepted_meta.append(synth_meta.iloc[:take].reset_index(drop=True))
            accepted_nn_distance.extend(dmin[:take].tolist())
            forbidden_hashes.update(hashes[:take])

        accepted_count = sum(len(a) for a in accepted)

        if accepted_count < need:
            print(f"  WARNING: accepted {accepted_count}/{need} novel synthetic rows.")

        if accepted_count:
            Xs = np.vstack(accepted)[:need]
            meta_s = pd.concat(accepted_meta, ignore_index=True).iloc[:len(Xs)]

            X_parts.append(Xs)
            y_parts.append(np.full(len(Xs), cls, dtype=int))
            meta_parts.append(meta_s.reset_index(drop=True))

        nn_arr = np.asarray(accepted_nn_distance, dtype=float)
        audit_rows.append({
            "Class": CLASS_NAMES[cls],
            "Real_Training_Rows": len(Xc),
            "Requested_Synthetic": need,
            "Accepted_Synthetic": min(accepted_count, need),
            "Generated_Candidates": generated_total,
            "Exact_Duplicate_Rate": exact_rejected / max(generated_total, 1),
            "Near_Duplicate_Rate": novelty_rejected / max(novelty_checked, 1),
            "NN_Distance_Min": float(nn_arr.min()) if len(nn_arr) else np.nan,
            "NN_Distance_Median": float(np.median(nn_arr)) if len(nn_arr) else np.nan,
            "NN_Distance_Mean": float(nn_arr.mean()) if len(nn_arr) else np.nan
        })

    X_aug = np.vstack(X_parts).astype(np.float32)
    y_aug = np.concatenate(y_parts).astype(int)
    meta_aug = pd.concat(meta_parts, ignore_index=True)

    order = rng.permutation(len(X_aug))
    X_aug = X_aug[order]
    y_aug = y_aug[order]
    meta_aug = meta_aug.iloc[order].reset_index(drop=True)
    y_bin_aug = (y_aug != 0).astype(int)

    wgan_gp_oversample.last_audit = pd.DataFrame(audit_rows)

    print("WGAN-GP counts after:", dict(sorted(Counter(y_aug.tolist()).items())))
    if len(audit_rows):
        print("\nWGAN-GP novelty audit")
        display(wgan_gp_oversample.last_audit.round(6))

    return X_aug, y_bin_aug, y_aug, meta_aug


wgan_gp_oversample.last_audit = pd.DataFrame()


## Block 11: GCN-GAT Graph Construction and Visualisation


In [ ]:
# Graph helpers.

def make_analysis_graph(X, meta, mapper, y=None, max_edges=GNN_EDGE_BUDGET, seed=SEED):
    X = np.asarray(X, dtype=np.float32)
    meta = meta.reset_index(drop=True)
    if y is None:
        use = np.arange(min(len(X), max_edges))
        yu = None
    else:
        y = np.asarray(y, dtype=int)
        use = stratified_budget_indices(y, max_edges, seed=seed)
        yu = y[use]

    graph = build_flow_graph(
        X[use],
        meta.iloc[use].reset_index(drop=True),
        mapper,
        y=yu,
        training=False,
        seed=seed
    )
    return graph

def visualise_flow_graph(graph, title="GCN + GAT Flow Graph", max_nodes=45, max_edges=120):
    ei = graph.edge_index.detach().cpu().numpy()
    edge_limit = min(max_edges, ei.shape[1])

    G = nx.DiGraph()
    for i in range(edge_limit):
        G.add_edge(int(ei[0, i]), int(ei[1, i]))

    if len(G.nodes()) == 0:
        print("No nodes available for graph visualization.")
        return

    degree = dict(G.degree())
    ranked = sorted(G.nodes(), key=lambda n: degree.get(n, 0), reverse=True)[:max_nodes]
    H = G.subgraph(ranked).copy()
    if len(H.nodes()) == 0:
        return

    node_degree = dict(H.degree())
    node_sizes = [350 + 120 * node_degree.get(n, 0) for n in H.nodes()]
    node_colors = [node_degree.get(n, 0) for n in H.nodes()]
    edge_colors = [node_degree.get(u, 0) for u, _ in H.edges()]

    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(H, seed=SEED, k=1.5, iterations=100)

    nx.draw_networkx_edges(
        H, pos, edge_color=edge_colors, edge_cmap=plt.cm.cool,
        width=1.8, alpha=0.45, arrows=True, arrowsize=14,
        connectionstyle="arc3,rad=0.10"
    )
    nodes = nx.draw_networkx_nodes(
        H, pos, node_size=node_sizes, node_color=node_colors,
        cmap=plt.cm.plasma, edgecolors="black", linewidths=1.2, alpha=0.95
    )
    nx.draw_networkx_labels(
        H, pos, font_size=8, font_color="white", font_weight="bold"
    )
    cbar = plt.colorbar(nodes, shrink=0.75)
    cbar.set_label("Node Degree")
    plt.title(title, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

def plot_degree_histogram(graph, title="Node Degree Distribution"):
    ei = graph.edge_index.detach().cpu().numpy()
    G = nx.DiGraph()
    G.add_edges_from(zip(ei[0].tolist(), ei[1].tolist()))
    degrees = [d for _, d in G.degree()]

    plt.figure(figsize=(8, 5))
    plt.hist(degrees, bins=min(30, max(5, len(set(degrees)))), edgecolor="black", alpha=0.85)
    plt.title(title, fontweight="bold")
    plt.xlabel("Node Degree")
    plt.ylabel("Count")
    plt.grid(axis="y", linestyle="--", alpha=0.30)
    plt.tight_layout()
    plt.show()


## Block 11B: Graph Statistics and Node-Degree Plots


In [ ]:
graph_stats_rows = []

def collect_graph_stats(graph, graph_name):
    labels = (
        graph.flow_y.detach().cpu().numpy()
        if hasattr(graph, "flow_y")
        else None
    )
    row = {
        "Graph": graph_name,
        "Nodes": int(graph.x.shape[0]),
        "Edges": int(graph.edge_index.shape[1]),
        "Node_Feature_Dim": int(graph.x.shape[1]),
        "Flow_Classes": int(len(np.unique(labels))) if labels is not None else np.nan
    }
    return row

def attack_graph_colored_by_class(graph, title="Training Attack Graph", max_edges=500, max_nodes=120):
    if not hasattr(graph, "flow_y"):
        print("Graph labels are unavailable.")
        return

    ei = graph.edge_index.detach().cpu().numpy()
    labels = graph.flow_y.detach().cpu().numpy().astype(int)
    n_edges = min(max_edges, ei.shape[1], len(labels))

    G = nx.Graph()
    selected_edges = [(int(ei[0, i]), int(ei[1, i])) for i in range(n_edges)]
    G.add_edges_from(selected_edges)
    if len(G.nodes()) == 0:
        return

    ranked = sorted(G.nodes(), key=lambda n: G.degree(n), reverse=True)[:max_nodes]
    H = G.subgraph(ranked).copy()

    node_class = {}
    for node in H.nodes():
        idxs = [
            i for i, (s, d) in enumerate(selected_edges)
            if s == node or d == node
        ]
        if idxs:
            vals, counts = np.unique(labels[idxs], return_counts=True)
            node_class[node] = int(vals[np.argmax(counts)])
        else:
            node_class[node] = 0

    pos = nx.spring_layout(H, seed=SEED, k=1.2, iterations=80)
    node_colors = [node_class[n] for n in H.nodes()]
    node_sizes = [240 + 70 * H.degree(n) for n in H.nodes()]

    plt.figure(figsize=(12, 8))
    nx.draw_networkx_edges(H, pos, alpha=0.28, width=1.2)
    sc = nx.draw_networkx_nodes(
        H, pos, node_color=node_colors, cmap="tab10",
        node_size=node_sizes, edgecolors="black", linewidths=0.8
    )
    nx.draw_networkx_labels(H, pos, font_size=7, font_color="white")
    cbar = plt.colorbar(sc, shrink=0.75)
    cbar.set_label("Dominant Class ID")
    plt.title(title, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

def node_degree_scatter(graph, title="Node-Degree Scatter Plot", max_edges=GNN_EDGE_BUDGET):
    if not hasattr(graph, "flow_y"):
        print("Graph labels are unavailable.")
        return

    ei = graph.edge_index.detach().cpu().numpy()
    labels = graph.flow_y.detach().cpu().numpy().astype(int)
    n_edges = min(max_edges, ei.shape[1], len(labels))

    G = nx.Graph()
    edges = [(int(ei[0, i]), int(ei[1, i])) for i in range(n_edges)]
    G.add_edges_from(edges)
    degree = dict(G.degree())

    src_degree = np.array([degree.get(int(ei[0, i]), 0) for i in range(n_edges)])
    dst_degree = np.array([degree.get(int(ei[1, i]), 0) for i in range(n_edges)])

    plt.figure(figsize=(8, 6))
    sc = plt.scatter(
        src_degree, dst_degree, c=labels[:n_edges],
        cmap="tab10", alpha=0.65, s=28, edgecolors="none"
    )
    cbar = plt.colorbar(sc)
    cbar.set_label("Class ID")
    plt.title(title, fontweight="bold")
    plt.xlabel("Source Node Degree")
    plt.ylabel("Destination Node Degree")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

def run_graph_analysis(final_output):
    mapper = final_output["upstream"].gnn_mapper

    graph_train = make_analysis_graph(
        final_output["X_aug"], final_output["meta_aug"], mapper,
        y=final_output["y_aug_mc"], max_edges=GNN_EDGE_BUDGET, seed=SEED
    )
    graph_test = make_analysis_graph(
        final_output["X_eval_selected"], final_output["meta_eval"], mapper,
        y=final_output["y_eval_mc"], max_edges=GNN_EDGE_BUDGET, seed=SEED + 1
    )
    graph_full = make_analysis_graph(
        final_output["X_fit_selected"], final_output["meta_fit"], mapper,
        y=final_output["y_fit_mc"], max_edges=GNN_EDGE_BUDGET, seed=SEED + 2
    )

    stats = pd.DataFrame([
        collect_graph_stats(graph_train, "Training/Augmented Graph"),
        collect_graph_stats(graph_test, "Test Graph"),
        collect_graph_stats(graph_full, "Full Training Graph")
    ])
    display(stats)

    visualise_flow_graph(graph_train, "GCN + GAT Flow Graph — Training Set")
    visualise_flow_graph(graph_test, "GCN + GAT Flow Graph — Test Set")
    plot_degree_histogram(graph_train, "Node Degree Distribution — Training Flow Graph")
    attack_graph_colored_by_class(graph_train, "Training Attack Graph Colored by Class")
    node_degree_scatter(graph_train, "Node-Degree Scatter Plot — Training Graph")
    node_degree_scatter(graph_test, "Node-Degree Scatter Plot — Test Graph")

    return {
        "train": graph_train,
        "test": graph_test,
        "full": graph_full,
        "stats": stats
    }


## Block 11C: Flow-Level and Packet-Level Feature Analysis


In [ ]:
FLOW_KEYWORDS = [
    "dur", "rate", "sload", "dload", "smean", "dmean",
    "ct_", "synack", "ackdat", "tcprtt"
]
PACKET_KEYWORDS = [
    "spkts", "dpkts", "sbytes", "dbytes", "sttl", "dttl",
    "sloss", "dloss", "swin", "dwin"
]

def _select_feature_group(columns, keywords):
    return [
        c for c in columns
        if any(k.lower() in c.lower() for k in keywords)
    ]

def _feature_group_summary(df, y_mc, features, group_name, top_n=12):
    rows = []
    y_mc = np.asarray(y_mc, dtype=int)

    for feature in [f for f in features if f in df.columns]:
        series = pd.to_numeric(df[feature], errors="coerce")
        normal = series[y_mc == 0].dropna()
        attack = series[y_mc != 0].dropna()

        if len(normal) == 0 or len(attack) == 0:
            continue

        rows.append({
            "Group": group_name,
            "Feature": feature,
            "Normal_Mean": normal.mean(),
            "Attack_Mean": attack.mean(),
            "Abs_Mean_Diff": abs(attack.mean() - normal.mean()),
            "Normal_Std": normal.std(),
            "Attack_Std": attack.std()
        })

    if not rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .sort_values("Abs_Mean_Diff", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

def run_feature_analysis(final_output, top_n=12):
    raw_train = final_output["fit_df"].reset_index(drop=True)
    y_train = final_output["y_fit_mc"]

    raw_columns = raw_feature_columns(raw_train)
    flow_features = _select_feature_group(raw_columns, FLOW_KEYWORDS)
    packet_features = _select_feature_group(raw_columns, PACKET_KEYWORDS)

    flow_summary = _feature_group_summary(
        raw_train, y_train, flow_features, "Flow-Level", top_n=top_n
    )
    packet_summary = _feature_group_summary(
        raw_train, y_train, packet_features, "Packet-Level", top_n=top_n
    )

    print("\nFlow-Level Feature Analysis")
    display(flow_summary.round(4))
    print("\nPacket-Level Feature Analysis")
    display(packet_summary.round(4))

    for title, table in [
        ("Flow-Level: Attack vs Normal Mean Difference", flow_summary),
        ("Packet-Level: Attack vs Normal Mean Difference", packet_summary)
    ]:
        if not table.empty:
            plt.figure(figsize=(10, 5))
            plt.barh(table["Feature"], table["Abs_Mean_Diff"])
            plt.title(title, fontweight="bold")
            plt.xlabel("Absolute Mean Difference")
            plt.gca().invert_yaxis()
            plt.tight_layout()
            plt.show()

    return pd.concat([flow_summary, packet_summary], ignore_index=True)


## Block 12: GCN-GAT Model


In [ ]:
class GraphMapper:
    def __init__(self):
        self.proto_to_id = None
        self.service_to_id = None
        self.unk_proto = None
        self.unk_service = None
        self.n_nodes = None

    def fit(self, meta):
        protos = sorted(meta["protocol"].astype(str).unique().tolist())
        services = sorted(meta["service"].astype(str).unique().tolist())

        self.proto_to_id = {p: i for i, p in enumerate(protos)}
        self.unk_proto = len(self.proto_to_id)

        service_offset = self.unk_proto + 1
        self.service_to_id = {
            s: service_offset + i for i, s in enumerate(services)
        }
        self.unk_service = service_offset + len(self.service_to_id)
        self.n_nodes = self.unk_service + 1
        return self

    def map(self, meta):
        src = np.array([
            self.proto_to_id.get(str(v), self.unk_proto)
            for v in meta["protocol"]
        ], dtype=np.int64)
        dst = np.array([
            self.service_to_id.get(str(v), self.unk_service)
            for v in meta["service"]
        ], dtype=np.int64)
        return src, dst

def stratified_budget_indices(y, budget, seed=SEED):
    y = np.asarray(y)
    if len(y) <= budget:
        return np.arange(len(y))

    rng = np.random.default_rng(seed)
    idx_out = []
    counts = Counter(y.tolist())

    for cls, cnt in sorted(counts.items()):
        n = max(1, int(round(budget * cnt / len(y))))
        idx = np.flatnonzero(y == cls)
        n = min(n, len(idx))
        idx_out.extend(rng.choice(idx, n, replace=False).tolist())

    if len(idx_out) > budget:
        idx_out = rng.choice(np.array(idx_out), budget, replace=False).tolist()
    elif len(idx_out) < budget:
        remain = np.setdiff1d(np.arange(len(y)), np.array(idx_out))
        extra = rng.choice(
            remain, min(budget - len(idx_out), len(remain)), replace=False
        )
        idx_out.extend(extra.tolist())

    return np.array(idx_out, dtype=int)

def build_flow_graph(X, meta, mapper, y=None, training=False, seed=SEED):
    X = np.asarray(X, dtype=np.float32)
    meta = meta.reset_index(drop=True)
    src_all, dst_all = mapper.map(meta)

    if training and y is not None:
        use = stratified_budget_indices(y, GNN_EDGE_BUDGET, seed)
    else:
        use = np.arange(len(X))

    Xu = X[use]
    src = src_all[use]
    dst = dst_all[use]
    yu = None if y is None else np.asarray(y, dtype=int)[use]

    node_sum = np.zeros((mapper.n_nodes, X.shape[1]), dtype=np.float32)
    node_cnt = np.zeros(mapper.n_nodes, dtype=np.float32)

    np.add.at(node_sum, src, Xu)
    np.add.at(node_sum, dst, Xu)
    np.add.at(node_cnt, src, 1.0)
    np.add.at(node_cnt, dst, 1.0)
    node_cnt = np.maximum(node_cnt, 1.0)

    node_features = node_sum / node_cnt[:, None]

    data = PyGData(
        x=torch.tensor(node_features, dtype=torch.float32),
        edge_index=torch.tensor(np.vstack([src, dst]), dtype=torch.long)
    )
    data.flow_src = torch.tensor(src, dtype=torch.long)
    data.flow_dst = torch.tensor(dst, dtype=torch.long)
    data.flow_indices = use

    if yu is not None:
        data.flow_y = torch.tensor(yu, dtype=torch.long)
    return data

class HybridGCNGAT(nn.Module):
    def __init__(self, in_ch, n_classes=NUM_CLASSES):
        super().__init__()
        self.gcn1 = GCNConv(in_ch, GNN_HIDDEN, add_self_loops=True)
        self.gat = GATConv(
            GNN_HIDDEN,
            GNN_HIDDEN // GNN_HEADS,
            heads=GNN_HEADS,
            dropout=GNN_DROPOUT,
            add_self_loops=True
        )
        self.gcn2 = GCNConv(GNN_HIDDEN, GNN_OUT, add_self_loops=True)
        self.drop = nn.Dropout(GNN_DROPOUT)
        self.classifier = nn.Linear(GNN_OUT, n_classes)
        self.pretrain_losses_ = []
        self.finetune_losses_ = []

    def node_embeddings(self, data):
        h = F.relu(self.gcn1(data.x, data.edge_index))
        h = self.drop(h)
        h = F.relu(self.gat(h, data.edge_index))
        h = self.drop(h)
        return F.relu(self.gcn2(h, data.edge_index))

    def flow_embeddings(self, data):
        h = self.node_embeddings(data)
        return (h[data.flow_src] + h[data.flow_dst]) / 2.0

    def forward(self, data):
        z = self.flow_embeddings(data)
        return self.classifier(z), z

def self_supervised_gnn_pretrain(model, graph, epochs=GNN_PRETRAIN_EPOCHS, lr=GNN_LR):
    decoder = nn.Linear(GNN_OUT, graph.x.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(
        list(model.parameters()) + list(decoder.parameters()), lr=lr
    )

    losses = []
    for ep in range(1, epochs + 1):
        model.train()
        decoder.train()
        optimizer.zero_grad()

        node_z = model.node_embeddings(graph)
        reconstructed = decoder(node_z)
        loss = F.mse_loss(reconstructed, graph.x)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.item()))

        if ep == 1 or ep % 20 == 0 or ep == epochs:
            print(f"  GNN pretrain {ep:03d}/{epochs}: reconstruction loss={loss.item():.6f}")

    model.pretrain_losses_ = losses
    return model

def fit_gnn_supervised(X, y_mc, meta, seed=SEED):
    X = np.asarray(X, dtype=np.float32)
    y_mc = np.asarray(y_mc, dtype=int)
    meta = meta.reset_index(drop=True)

    idx = np.arange(len(X))
    tr_idx, va_idx = train_test_split(
        idx, test_size=0.15, random_state=seed, stratify=y_mc
    )

    mapper = GraphMapper().fit(meta.iloc[tr_idx].reset_index(drop=True))

    tr_graph = build_flow_graph(
        X[tr_idx],
        meta.iloc[tr_idx].reset_index(drop=True),
        mapper,
        y=y_mc[tr_idx],
        training=True,
        seed=seed
    ).to(DEVICE)

    va_graph = build_flow_graph(
        X[va_idx],
        meta.iloc[va_idx].reset_index(drop=True),
        mapper,
        y=y_mc[va_idx],
        training=False,
        seed=seed
    ).to(DEVICE)

    model = HybridGCNGAT(X.shape[1]).to(DEVICE)
    print("\nSelf-supervised GNN pre-training")
    self_supervised_gnn_pretrain(model, tr_graph)

    y_budget = tr_graph.flow_y.detach().cpu().numpy()
    classes_present = np.unique(y_budget)
    weights = np.ones(NUM_CLASSES, dtype=np.float32)
    cw = compute_class_weight(
        class_weight="balanced",
        classes=classes_present,
        y=y_budget
    )
    for c, w in zip(classes_present, cw):
        weights[int(c)] = float(w)

    weight_t = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=GNN_LR)

    best_state = None
    best_loss = np.inf
    best_epoch = 1
    train_losses = []

    print("Supervised GNN fine-tuning")
    for ep in range(1, GNN_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        logits, _ = model(tr_graph)
        loss = F.cross_entropy(logits, tr_graph.flow_y, weight=weight_t)
        loss.backward()
        optimizer.step()
        train_losses.append(float(loss.item()))

        model.eval()
        with torch.no_grad():
            v_logits, _ = model(va_graph)
            v_loss = F.cross_entropy(
                v_logits, va_graph.flow_y, weight=weight_t
            ).item()

        if v_loss < best_loss - 1e-5:
            best_loss = v_loss
            best_epoch = ep
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

    model.finetune_losses_ = train_losses
    print(
        f"GNN selected epoch: {best_epoch}, "
        f"validation CE={best_loss:.5f}"
    )

    mapper_final = GraphMapper().fit(meta)
    full_graph = build_flow_graph(
        X, meta, mapper_final, y=y_mc,
        training=True, seed=seed
    ).to(DEVICE)

    model_final = HybridGCNGAT(X.shape[1]).to(DEVICE)
    print("Self-supervised pre-training on the complete training graph")
    self_supervised_gnn_pretrain(model_final, full_graph)

    y_budget = full_graph.flow_y.detach().cpu().numpy()
    classes_present = np.unique(y_budget)
    weights = np.ones(NUM_CLASSES, dtype=np.float32)
    cw = compute_class_weight(
        class_weight="balanced",
        classes=classes_present,
        y=y_budget
    )
    for c, w in zip(classes_present, cw):
        weights[int(c)] = float(w)

    weight_t = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
    optimizer = torch.optim.Adam(model_final.parameters(), lr=GNN_LR)

    final_losses = []
    for ep in range(1, best_epoch + 1):
        model_final.train()
        optimizer.zero_grad()
        logits, _ = model_final(full_graph)
        loss = F.cross_entropy(logits, full_graph.flow_y, weight=weight_t)
        loss.backward()
        optimizer.step()
        final_losses.append(float(loss.item()))

    model_final.finetune_losses_ = final_losses
    return mapper_final, model_final

def gnn_transform(X, meta, mapper, model):
    graph = build_flow_graph(
        X, meta, mapper, y=None, training=False
    ).to(DEVICE)

    model.eval()
    with torch.no_grad():
        z = model.flow_embeddings(graph).cpu().numpy()

    return z.astype(np.float32)


## Block 13: MFV-NSA


In [ ]:

def choose_binary_threshold(y_true, scores):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    best_t, best_f = 0.5, -1.0
    for t in np.linspace(0.01, 0.99, 99):
        pred = (scores >= t).astype(int)
        f = f1_score(y_true, pred, average="binary", zero_division=0)
        if f > best_f:
            best_t, best_f = float(t), float(f)
    return best_t


membership_behavior_reference = pd.DataFrame([
    {
        "Distance regime": "Near the self-set",
        "Condition": "d_self << r_boundary",
        "Distance membership": "mu_dist ≈ 0"
    },
    {
        "Distance regime": "At the adaptive boundary",
        "Condition": "d_self = r_boundary",
        "Distance membership": "mu_dist = 0.5"
    },
    {
        "Distance regime": "Far from the self-set",
        "Condition": "d_self >> r_boundary",
        "Distance membership": "mu_dist ≈ 1"
    }
])

immune_feature_schema = pd.DataFrame([
    ("Nearest-self distance", "d_self"),
    ("Fuzzy anomaly score", "s_fuzzy"),
    ("Distance membership", "mu_dist"),
    ("Detector-activation membership", "mu_det"),
    ("Minimum detector distance", "d_min"),
    ("Mean detector distance", "d_mean"),
    ("Detector-distance deviation", "sigma_d"),
    ("Minimum activation radius", "nu_min"),
    ("Mean activation radius", "nu_mean"),
    ("Activation-radius deviation", "sigma_nu"),
    ("Activated-detector count", "act_K"),
    ("Maximum detector penetration", "p_max"),
    ("Mean detector penetration", "p_mean"),
    ("Decision margin", "s_fuzzy - tau"),
], columns=["Feature", "Symbol"])

print("Distance-membership behaviour")
display(membership_behavior_reference)

print("\n14-dimensional MFV-NSA feature schema")
display(immune_feature_schema)


class MFVNSA:
    def __init__(self, seed=SEED):
        self.seed = seed
        self.self_set = None
        self.self_tree = None
        self.r_boundary = None
        self.detectors = np.empty((0, 0), dtype=np.float32)
        self.radii = np.empty((0,), dtype=np.float32)
        self.det_tree = None
        self.tau = 0.5

    # Stage 1: self-set
    def construct_self_set(self, X, y_bin):
        X = np.asarray(X, dtype=np.float32)
        y_bin = np.asarray(y_bin, dtype=int)

        S = X[y_bin == 0]
        S = np.unique(S, axis=0)  # exact duplicates removed

        if len(S) < 2:
            raise RuntimeError("Insufficient benign training samples for MFV-NSA.")

        rng = np.random.default_rng(self.seed)
        if len(S) > NSA_SELF_MAX:
            idx = rng.choice(len(S), NSA_SELF_MAX, replace=False)
            S = S[idx]

        self.self_set = S.astype(np.float32)
        self.self_tree = BallTree(self.self_set, metric="euclidean")
        return self

    # Stage 2: adaptive boundary
    def estimate_adaptive_boundary(self):
        if self.self_tree is None or self.self_set is None:
            raise RuntimeError("Construct the self-set before estimating the boundary.")

        two_dist, _ = self.self_tree.query(self.self_set, k=2)
        nearest_distinct_self = two_dist[:, 1]

        self.r_boundary = max(
            float(np.percentile(nearest_distinct_self, NSA_RHO)),
            1e-12
        )
        return self.r_boundary

    # Candidate pool.
    def sample_candidate_pool(self, X):
        X = np.asarray(X, dtype=np.float32)
        rng = np.random.default_rng(self.seed)

        n, d = X.shape
        lo = X.min(axis=0)
        hi = X.max(axis=0)
        span = np.maximum(hi - lo, 1e-6)

        # Training-space candidates.
        n_local = int(0.70 * NSA_N_CAND)
        anchors = X[rng.integers(0, n, size=n_local)]
        local = anchors + rng.normal(0.0, 0.05, size=(n_local, d)) * span
        local = np.clip(local, lo, hi)

        n_uniform = NSA_N_CAND - n_local
        uniform = rng.uniform(lo, hi, size=(n_uniform, d))

        pool = np.vstack([local, uniform]).astype(np.float32)
        rng.shuffle(pool)
        return pool

    # Stage 3: constrained detectors
    def generate_detectors(self, X):
        if self.self_tree is None:
            raise RuntimeError("Self-set spatial index is not available.")

        candidate_pool = self.sample_candidate_pool(X)
        dets, radii = [], []

        for candidate in candidate_pool:
            if len(dets) >= NSA_N_DET:
                break

            # Distance-to-self constraint.
            d_self = float(self.self_tree.query(candidate.reshape(1, -1), k=1)[0][0, 0])
            if d_self < NSA_R_MIN or d_self > NSA_R_MAX:
                continue

            r_c = d_self - NSA_DELTA
            if r_c <= 0:
                continue

            # Detector-separation constraint.
            valid = True
            if dets:
                D = np.asarray(dets, dtype=np.float32)
                R = np.asarray(radii, dtype=np.float32)
                center_distance = np.linalg.norm(D - candidate, axis=1)
                if np.any(center_distance < NSA_BETA * np.minimum(r_c, R)):
                    valid = False

            # Accept valid detector.
            if valid:
                dets.append(candidate.copy())
                radii.append(float(r_c))

        d = np.asarray(dets, dtype=np.float32)
        if d.size == 0:
            d = np.empty((0, np.asarray(X).shape[1]), dtype=np.float32)

        self.detectors = d
        self.radii = np.asarray(radii, dtype=np.float32)
        self.det_tree = (
            BallTree(self.detectors, metric="euclidean")
            if len(self.detectors) else None
        )

        print(f"MFV-NSA accepted detectors: {len(self.detectors)}/{NSA_N_DET}")
        return self

    def fit_core(self, X, y_bin):
        self.construct_self_set(X, y_bin)
        self.estimate_adaptive_boundary()
        self.generate_detectors(X)
        return self

    # Stage 4: fuzzy score
    def fuzzy_components(self, X):
        X = np.asarray(X, dtype=np.float32)

        d_self, _ = self.self_tree.query(X, k=1)
        d_self = d_self.ravel()

        mu_dist = 1.0 / (
            1.0 + np.exp(
                -NSA_GAMMA * (d_self - self.r_boundary)
            )
        )

        if self.det_tree is None or len(self.detectors) == 0:
            mu_det = np.zeros(len(X), dtype=np.float64)
            s_fuzzy = np.zeros(len(X), dtype=np.float64)
            return s_fuzzy, mu_dist, mu_det, d_self

        k_prime = min(NSA_K, len(self.detectors))
        detector_distance, detector_index = self.det_tree.query(X, k=k_prime)

        activated = detector_distance <= self.radii[detector_index]
        mu_det = activated.sum(axis=1) / float(k_prime)

        # Harmonic fusion.
        s_fuzzy = (
            (NSA_W_DIST + NSA_W_DET)
            / (
                NSA_W_DIST / (mu_dist + NSA_EPS)
                + NSA_W_DET / (mu_det + NSA_EPS)
            )
        )
        s_fuzzy = np.minimum(1.0, s_fuzzy)

        return s_fuzzy, mu_dist, mu_det, d_self

    def tune_tau(self, X_val, y_val_bin):
        score = self.fuzzy_components(X_val)[0]
        self.tau = choose_binary_threshold(y_val_bin, score)
        print(f"MFV-NSA selected tau={self.tau:.3f}")
        return self.tau

    # Stage 5: 14-D feature vector
    def transform(self, X):
        X = np.asarray(X, dtype=np.float32)
        s_fuzzy, mu_dist, mu_det, d_self = self.fuzzy_components(X)
        n = len(X)

        if self.det_tree is None or len(self.detectors) == 0:
            Z = np.column_stack([
                d_self, s_fuzzy, mu_dist, mu_det,
                np.zeros(n), np.zeros(n), np.zeros(n),
                np.zeros(n), np.zeros(n), np.zeros(n),
                np.zeros(n), np.zeros(n), np.zeros(n),
                s_fuzzy - self.tau
            ])
            return Z.astype(np.float32)

        k_prime = min(NSA_K, len(self.detectors))
        detector_distance, detector_index = self.det_tree.query(X, k=k_prime)

        # Detector distances.
        d_min = detector_distance.min(axis=1)
        d_mean = detector_distance.mean(axis=1)
        sigma_d = detector_distance.std(axis=1)

        # Activated detectors.
        active = detector_distance <= self.radii[detector_index]

        nu_min = np.zeros(n)
        nu_mean = np.zeros(n)
        sigma_nu = np.zeros(n)
        act_K = active.sum(axis=1).astype(float)
        p_max = np.zeros(n)
        p_mean = np.zeros(n)

        for i in range(n):
            if active[i].any():
                active_radii = self.radii[detector_index[i][active[i]]]
                active_distance = detector_distance[i][active[i]]
                penetration = active_radii - active_distance

                nu_min[i] = active_radii.min()
                nu_mean[i] = active_radii.mean()
                sigma_nu[i] = active_radii.std()
                p_max[i] = penetration.max()
                p_mean[i] = penetration.mean()

        margin = s_fuzzy - self.tau

        Z = np.column_stack([
            d_self,
            s_fuzzy,
            mu_dist,
            mu_det,
            d_min,
            d_mean,
            sigma_d,
            nu_min,
            nu_mean,
            sigma_nu,
            act_K,
            p_max,
            p_mean,
            margin
        ])

        if Z.shape[1] != 14:
            raise RuntimeError(f"MFV-NSA feature dimension is {Z.shape[1]}, expected 14.")
        return Z.astype(np.float32)

    def score(self, X):
        return self.fuzzy_components(X)[0]


## Block 14: CSA Multi-Class Head


In [ ]:
class ClonalSelectionClassifier:
    def __init__(self, seed=SEED):
        self.seed = seed
        self.cells = None
        self.cell_labels = None

    @staticmethod
    def _affinity(cell, X, y, cls):
        d = np.linalg.norm(
            X - cell, axis=1
        )

        radius = np.percentile(
            d, 30
        )

        nbr = d <= radius

        if nbr.sum() == 0:
            return 0.0

        return float(
            np.mean(y[nbr] == cls)
        )

    def fit(self, X, y):
        X = np.asarray(
            X, dtype=np.float32
        )
        y = np.asarray(
            y, dtype=int
        )

        rng = np.random.default_rng(
            self.seed
        )

        cells_all = []
        labels_all = []

        for cls in sorted(
            np.unique(y)
        ):
            Xc = X[y == cls]
            if len(Xc) == 0:
                continue

            center = Xc.mean(
                axis=0
            )

            k = min(
                CSA_N_MEM,
                len(Xc)
            )

            mem = Xc[
                np.argsort(
                    np.linalg.norm(
                        Xc - center,
                        axis=1
                    )
                )[:k]
            ].copy()

            for _ in range(
                CSA_N_GEN
            ):
                clones = []

                for c in mem:
                    for _ in range(
                        CSA_N_CLONES
                    ):
                        clones.append(
                            c + rng.normal(
                                0.0,
                                CSA_ETA,
                                size=c.shape
                            )
                        )

                clones = np.asarray(
                    clones,
                    dtype=np.float32
                )

                aff = np.array([
                    self._affinity(
                        c,
                        X,
                        y,
                        cls
                    )
                    for c in clones
                ])

                mem = clones[
                    np.argsort(
                        aff
                    )[-k:]
                ]

            cells_all.append(
                mem
            )

            labels_all.extend(
                [cls] * len(mem)
            )

        self.cells = np.vstack(
            cells_all
        ).astype(np.float32)

        self.cell_labels = np.asarray(
            labels_all,
            dtype=int
        )

        return self

    def predict_proba(self, X):
        X = np.asarray(
            X, dtype=np.float32
        )

        P = np.zeros(
            (len(X), NUM_CLASSES),
            dtype=np.float64
        )

        for i, x in enumerate(X):
            d = np.linalg.norm(
                self.cells - x,
                axis=1
            )

            for cls in range(
                NUM_CLASSES
            ):
                m = (
                    self.cell_labels
                    == cls
                )

                if m.any():
                    P[i, cls] = (
                        1.0
                        / (
                            d[m].min()
                            + 1e-9
                        )
                    )

        P /= (
            P.sum(
                axis=1,
                keepdims=True
            )
            + 1e-12
        )

        return P.astype(
            np.float32
        )

    def predict(self, X):
        return (
            self.predict_proba(X)
            .argmax(axis=1)
        )


## Block 15: Shared Embedding


In [ ]:
def assemble_shared_embedding(
    X_sel,
    Z_nsa,
    Z_gnn,
    Z_csa
):
    return np.hstack([
        np.asarray(
            X_sel,
            dtype=np.float32
        ),
        np.asarray(
            Z_nsa,
            dtype=np.float32
        ),
        np.asarray(
            Z_gnn,
            dtype=np.float32
        ),
        np.asarray(
            Z_csa,
            dtype=np.float32
        )
    ]).astype(np.float32)


## Block 16: CNN-BiLSTM-MHA


In [ ]:
def build_cnn_bilstm_mha(
    input_dim
):
    inp = Input(
        shape=(input_dim, 1),
        name="shared_embedding"
    )

    x = Conv1D(
        CNN_CONV_FILTERS[0],
        3,
        padding="same",
        activation="relu"
    )(inp)

    x = BatchNormalization()(x)
    x = Dropout(CNN_DROPOUTS[0])(x)

    x = Conv1D(
        CNN_CONV_FILTERS[1],
        3,
        padding="same",
        activation="relu"
    )(x)

    x = BatchNormalization()(x)

    h = Bidirectional(
        LSTM(
            CNN_BILSTM_UNITS,
            return_sequences=True
        )
    )(x)

    attn = MultiHeadAttention(
        num_heads=CNN_MHA_HEADS,
        key_dim=CNN_MHA_KEY_DIM
    )(h, h)

    m = LayerNormalization()(
        Add()([h, attn])
    )

    r = GlobalAveragePooling1D()(
        m
    )

    z = Dense(
        CNN_DENSE_UNITS[0],
        activation="relu"
    )(r)

    z = Dropout(
        CNN_DROPOUTS[1]
    )(z)

    z = Dense(
        CNN_DENSE_UNITS[1],
        activation="relu"
    )(z)

    z = Dropout(
        CNN_DROPOUTS[2]
    )(z)

    binary = Dense(
        1,
        activation="sigmoid",
        name="binary"
    )(z)

    multiclass = Dense(
        NUM_CLASSES,
        activation="softmax",
        name="multiclass"
    )(z)

    model = Model(
        inp,
        [
            binary,
            multiclass
        ]
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            CNN_LR
        ),
        loss={
            "binary":
            "binary_crossentropy",
            "multiclass":
            "sparse_categorical_crossentropy"
        },
        loss_weights={
            "binary": 1.0,
            "multiclass": 0.5
        },
        metrics={
            "binary": ["accuracy"],
            "multiclass": ["accuracy"]
        }
    )

    return model

def seqify(Z):
    Z = np.asarray(
        Z,
        dtype=np.float32
    )

    return Z.reshape(
        len(Z),
        Z.shape[1],
        1
    )


## Block 17: Cross-Fitted Upstream Features


In [ ]:
def crossfit_upstream(
    X,
    y_bin,
    y_mc,
    meta,
    groups,
    selected_feature_names,
    categorical_names,
    seed=SEED
):
    X = np.asarray(
        X,
        dtype=np.float32
    )

    y_bin = np.asarray(
        y_bin,
        dtype=int
    )

    y_mc = np.asarray(
        y_mc,
        dtype=int
    )

    groups = np.asarray(
        groups
    ).astype(str)

    meta = meta.reset_index(
        drop=True
    )

    Z_nsa = np.zeros(
        (len(X), 14),
        dtype=np.float32
    )

    Z_gnn = np.zeros(
        (len(X), GNN_OUT),
        dtype=np.float32
    )

    Z_csa = np.zeros(
        (len(X), NUM_CLASSES),
        dtype=np.float32
    )

    splitter = StratifiedGroupKFold(
        n_splits=INNER_CF_FOLDS,
        shuffle=True,
        random_state=seed
    )

    for q, (tri, vai) in enumerate(
        splitter.split(
            X,
            y_mc,
            groups
        ),
        1
    ):
        print(
            f"\n--- Upstream cross-fit "
            f"{q}/{INNER_CF_FOLDS} ---"
        )

        Xa, yba, yma, ma = (
            wgan_gp_oversample(
                X[tri],
                y_mc[tri],
                meta.iloc[
                    tri
                ].reset_index(
                    drop=True
                ),
                selected_feature_names,
                categorical_names,
                forbidden_arrays=[
                    X[vai]
                ],
                seed=seed + q
            )
        )

        nsa = MFVNSA(
            seed=seed + q
        ).fit_core(
            Xa,
            yba
        )

        # Tune threshold on the fit folds only.
        nsa.tau = choose_binary_threshold(
            y_bin[tri],
            nsa.score(X[tri])
        )

        Z_nsa[vai] = (
            nsa.transform(
                X[vai]
            )
        )

        mapper, gnn = (
            fit_gnn_supervised(
                Xa,
                yma,
                ma,
                seed=seed + q
            )
        )

        Z_gnn[vai] = (
            gnn_transform(
                X[vai],
                meta.iloc[
                    vai
                ].reset_index(
                    drop=True
                ),
                mapper,
                gnn
            )
        )

        csa = (
            ClonalSelectionClassifier(
                seed=seed + q
            )
            .fit(
                Xa,
                yma
            )
        )

        Z_csa[vai] = (
            csa.predict_proba(
                X[vai]
            )
        )

    return (
        Z_nsa,
        Z_gnn,
        Z_csa
    )

@dataclass
class FinalUpstream:
    nsa: object
    gnn_mapper: object
    gnn_model: object
    csa: object

def fit_final_upstream(
    X_aug,
    y_bin_aug,
    y_mc_aug,
    meta_aug,
    seed=SEED
):
    nsa = MFVNSA(
        seed=seed
    ).fit_core(
        X_aug,
        y_bin_aug
    )

    mapper, gnn = (
        fit_gnn_supervised(
            X_aug,
            y_mc_aug,
            meta_aug,
            seed=seed
        )
    )

    csa = (
        ClonalSelectionClassifier(
            seed=seed
        )
        .fit(
            X_aug,
            y_mc_aug
        )
    )

    return FinalUpstream(
        nsa,
        mapper,
        gnn,
        csa
    )

def transform_upstream(
    X,
    meta,
    upstream
):
    Z_nsa = (
        upstream.nsa
        .transform(
            X
        )
    )

    Z_gnn = gnn_transform(
        X,
        meta,
        upstream.gnn_mapper,
        upstream.gnn_model
    )

    Z_csa = (
        upstream.csa
        .predict_proba(
            X
        )
    )

    return (
        Z_nsa,
        Z_gnn,
        Z_csa
    )


## Block 18: Decision Fusion


In [ ]:
def clip_prob(p):
    return np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        1e-6,
        1.0 - 1e-6
    )

def logit(p):
    p = clip_prob(p)
    return np.log(
        p / (1.0 - p)
    )

class ScalarPlatt:
    def __init__(self):
        self.model = LogisticRegression(
            max_iter=1000,
            random_state=SEED
        )

    def fit(self, p, y):
        self.model.fit(
            logit(p).reshape(-1, 1),
            y
        )
        return self

    def transform(self, p):
        return (
            self.model
            .predict_proba(
                logit(p).reshape(-1, 1)
            )[:, 1]
        )

class MulticlassLogitCalibrator:
    def __init__(self):
        self.model = LogisticRegression(
            max_iter=2000,
            random_state=SEED
        )

    def _X(self, P):
        P = np.clip(
            np.asarray(
                P,
                dtype=float
            ),
            1e-8,
            1.0
        )

        P = (
            P
            / (
                P.sum(
                    axis=1,
                    keepdims=True
                )
                + 1e-12
            )
        )

        return np.log(P)

    def fit(self, P, y):
        self.model.fit(
            self._X(P),
            y
        )
        return self

    def transform(self, P):
        out = (
            self.model
            .predict_proba(
                self._X(P)
            )
        )

        full = np.zeros(
            (
                len(P),
                NUM_CLASSES
            ),
            dtype=float
        )

        for j, c in enumerate(
            self.model.classes_
        ):
            full[:, int(c)] = (
                out[:, j]
            )

        full /= (
            full.sum(
                axis=1,
                keepdims=True
            )
            + 1e-12
        )

        return full

def binary_fpr(
    y,
    pred
):
    tn, fp, fn, tp = (
        confusion_matrix(
            y,
            pred,
            labels=[0, 1]
        ).ravel()
    )

    return (
        fp
        / max(
            fp + tn,
            1
        )
    )

class CalibratedDecisionFusion:
    def __init__(self):
        self.c_nsa = ScalarPlatt()
        self.c_bin = ScalarPlatt()
        self.c_csa = ScalarPlatt()

        self.stack = LogisticRegression(
            max_iter=1000,
            random_state=SEED
        )

        self.c_final_mc = (
            MulticlassLogitCalibrator()
        )

        self.c_csa_mc = (
            MulticlassLogitCalibrator()
        )

        self.theta_fuse = 0.5
        self.lambda_mc = 0.5
        self.tau_reject = 0.5

    def _calibrated_binary_matrix(
        self,
        nsa_score,
        cnn_bin,
        csa_p
    ):
        pn = self.c_nsa.transform(
            nsa_score
        )

        pb = self.c_bin.transform(
            cnn_bin
        )

        pc = self.c_csa.transform(
            1.0 - csa_p[:, 0]
        )

        Xs = np.column_stack([
            logit(pn),
            logit(pb),
            logit(pc)
        ])

        return (
            pn,
            pb,
            pc,
            Xs
        )

    def fit(
        self,
        nsa_score,
        cnn_bin,
        cnn_mc,
        csa_p,
        y_bin,
        y_mc
    ):
        y_bin = np.asarray(
            y_bin,
            dtype=int
        )

        y_mc = np.asarray(
            y_mc,
            dtype=int
        )

        self.c_nsa.fit(
            nsa_score,
            y_bin
        )

        self.c_bin.fit(
            cnn_bin,
            y_bin
        )

        self.c_csa.fit(
            1.0 - csa_p[:, 0],
            y_bin
        )

        _, _, _, Xs = (
            self._calibrated_binary_matrix(
                nsa_score,
                cnn_bin,
                csa_p
            )
        )

        self.stack.fit(
            Xs,
            y_bin
        )

        pf = (
            self.stack
            .predict_proba(
                Xs
            )[:, 1]
        )

        best = None

        for t in np.linspace(
            0.05,
            0.95,
            91
        ):
            pred = (
                pf >= t
            ).astype(int)

            f = f1_score(
                y_bin,
                pred,
                average="macro",
                zero_division=0
            )

            fp = binary_fpr(
                y_bin,
                pred
            )

            key = (
                f,
                -fp
            )

            if (
                best is None
                or key > best[0]
            ):
                best = (
                    key,
                    float(t)
                )

        self.theta_fuse = (
            best[1]
        )

        self.c_final_mc.fit(
            cnn_mc,
            y_mc
        )

        self.c_csa_mc.fit(
            csa_p,
            y_mc
        )

        p_final = (
            self.c_final_mc
            .transform(
                cnn_mc
            )
        )

        p_csa = (
            self.c_csa_mc
            .transform(
                csa_p
            )
        )

        best_f = -1.0
        best_lam = 0.5

        attack_mask = (
            y_mc > 0
        )

        for lam in np.linspace(
            0.0,
            1.0,
            21
        ):
            q = (
                lam * p_final
                + (
                    1.0 - lam
                ) * p_csa
            )

            pred_attack = (
                1
                + np.argmax(
                    q[:, 1:],
                    axis=1
                )
            )

            if attack_mask.any():
                f = f1_score(
                    y_mc[
                        attack_mask
                    ],
                    pred_attack[
                        attack_mask
                    ],
                    average="macro",
                    zero_division=0
                )
            else:
                f = 0.0

            if f > best_f:
                best_f = f
                best_lam = float(
                    lam
                )

        self.lambda_mc = (
            best_lam
        )

        q = (
            self.lambda_mc
            * p_final
            + (
                1.0
                - self.lambda_mc
            ) * p_csa
        )

        conf = q[:, 1:].max(
            axis=1
        )

        attack_cls = (
            1
            + np.argmax(
                q[:, 1:],
                axis=1
            )
        )

        best_f = -1.0
        best_tau = 0.5

        for tau in np.linspace(
            0.10,
            0.95,
            86
        ):
            pred = np.where(
                pf
                < self.theta_fuse,
                0,
                np.where(
                    conf < tau,
                    -1,
                    attack_cls
                )
            )

            f = f1_score(
                y_mc,
                pred,
                labels=list(
                    range(
                        NUM_CLASSES
                    )
                ),
                average="macro",
                zero_division=0
            )

            if f > best_f:
                best_f = f
                best_tau = float(
                    tau
                )

        self.tau_reject = (
            best_tau
        )

        print(
            "Fusion parameters: "
            f"theta_fuse="
            f"{self.theta_fuse:.3f}, "
            f"lambda="
            f"{self.lambda_mc:.2f}, "
            f"tau_reject="
            f"{self.tau_reject:.3f}"
        )

        return self

    def predict(
        self,
        nsa_score,
        cnn_bin,
        cnn_mc,
        csa_p
    ):
        _, _, _, Xs = (
            self._calibrated_binary_matrix(
                nsa_score,
                cnn_bin,
                csa_p
            )
        )

        pf = (
            self.stack
            .predict_proba(
                Xs
            )[:, 1]
        )

        p_final = (
            self.c_final_mc
            .transform(
                cnn_mc
            )
        )

        p_csa = (
            self.c_csa_mc
            .transform(
                csa_p
            )
        )

        q = (
            self.lambda_mc
            * p_final
            + (
                1.0
                - self.lambda_mc
            ) * p_csa
        )

        conf = q[:, 1:].max(
            axis=1
        )

        attack_cls = (
            1
            + np.argmax(
                q[:, 1:],
                axis=1
            )
        )

        pred_bin = (
            pf
            >= self.theta_fuse
        ).astype(int)

        pred_mc = np.where(
            pred_bin == 0,
            0,
            np.where(
                conf
                < self.tau_reject,
                -1,
                attack_cls
            )
        )

        return (
            pred_bin,
            pred_mc.astype(int),
            pf,
            q
        )


## Block 19: Metrics and Fuzzy Scores


In [ ]:
def binary_metrics(
    y_true,
    y_pred
):
    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int
    )

    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1]
        ).ravel()
    )

    return {
        "Accuracy":
        accuracy_score(
            y_true,
            y_pred
        ),

        "Precision":
        precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "Recall":
        recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "F1":
        f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "FPR":
        fp
        / max(
            fp + tn,
            1
        ),

        "TNR":
        tn
        / max(
            tn + fp,
            1
        ),

        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn)
    }

def multiclass_metrics(
    y_true,
    y_pred
):
    y_true = np.asarray(
        y_true,
        dtype=int
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int
    )

    fprs = []
    tnrs = []

    for c in range(
        NUM_CLASSES
    ):
        yt = (
            y_true == c
        ).astype(int)

        yp = (
            y_pred == c
        ).astype(int)

        tn, fp, fn, tp = (
            confusion_matrix(
                yt,
                yp,
                labels=[0, 1]
            ).ravel()
        )

        fprs.append(
            fp
            / max(
                fp + tn,
                1
            )
        )

        tnrs.append(
            tn
            / max(
                tn + fp,
                1
            )
        )

    labels = list(
        range(
            NUM_CLASSES
        )
    )

    return {
        "Accuracy":
        accuracy_score(
            y_true,
            y_pred
        ),

        "Precision":
        precision_score(
            y_true,
            y_pred,
            labels=labels,
            average="macro",
            zero_division=0
        ),

        "Recall":
        recall_score(
            y_true,
            y_pred,
            labels=labels,
            average="macro",
            zero_division=0
        ),

        "F1":
        f1_score(
            y_true,
            y_pred,
            labels=labels,
            average="macro",
            zero_division=0
        ),

        "FPR":
        float(
            np.mean(
                fprs
            )
        ),

        "TNR":
        float(
            np.mean(
                tnrs
            )
        ),

        "Rejected":
        int(
            np.sum(
                y_pred == -1
            )
        )
    }

def plot_fuzzy_anomaly_distribution(
    X_eval,
    y_bin_eval,
    nsa,
    title="UNSW-NB15"
):
    sf, mu_dist, mu_det, d_self = (
        nsa.fuzzy_components(
            X_eval
        )
    )

    y = np.asarray(
        y_bin_eval,
        dtype=int
    )

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15, 4)
    )

    axes[0].hist(
        d_self[y == 0],
        bins=60,
        density=True,
        alpha=0.55,
        label="Normal"
    )

    axes[0].hist(
        d_self[y == 1],
        bins=60,
        density=True,
        alpha=0.55,
        label="Attack"
    )

    axes[0].axvline(
        nsa.r_boundary,
        linestyle="--",
        linewidth=2,
        label=(
            f"Boundary="
            f"{nsa.r_boundary:.3f}"
        )
    )

    axes[0].set_title(
        "Distance from benign self-set"
    )
    axes[0].set_xlabel(
        "Nearest-self distance"
    )
    axes[0].set_ylabel(
        "Density"
    )
    axes[0].legend()

    order = np.argsort(
        d_self
    )

    axes[1].plot(
        d_self[order],
        mu_dist[order],
        linewidth=2
    )

    axes[1].axvline(
        nsa.r_boundary,
        linestyle="--",
        linewidth=2
    )

    axes[1].axhline(
        0.5,
        linestyle=":",
        linewidth=1.5
    )

    axes[1].set_title(
        "Adaptive fuzzy membership"
    )
    axes[1].set_xlabel(
        "Nearest-self distance"
    )
    axes[1].set_ylabel(
        "mu_dist"
    )

    axes[2].hist(
        sf[y == 0],
        bins=60,
        density=True,
        alpha=0.55,
        label="Normal"
    )

    axes[2].hist(
        sf[y == 1],
        bins=60,
        density=True,
        alpha=0.55,
        label="Attack"
    )

    axes[2].axvline(
        nsa.tau,
        linestyle="--",
        linewidth=2,
        label=(
            f"Validation tau="
            f"{nsa.tau:.3f}"
        )
    )

    axes[2].set_title(
        "Harmonic-fusion anomaly score"
    )

    axes[2].set_xlabel(
        "Fuzzy anomaly score"
    )

    axes[2].set_ylabel(
        "Density"
    )

    axes[2].legend()

    plt.suptitle(
        f"MFV-NSA Fuzzy Anomaly "
        f"Score Analysis — {title}",
        fontweight="bold"
    )

    plt.tight_layout()
    plt.show()


def binary_curve_metrics(y_true, y_pred, y_prob):
    out = binary_metrics(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    pr_precision, pr_recall, _ = precision_recall_curve(y_true, y_prob)
    out["ROC_AUC"] = auc(fpr, tpr)
    out["PR_AUC"] = auc(pr_recall, pr_precision)
    out["roc_fpr"] = fpr
    out["roc_tpr"] = tpr
    out["pr_precision"] = pr_precision
    out["pr_recall"] = pr_recall
    return out

def shared_feature_names(selected_names):
    immune_names = [
        "nearest_self_distance", "fuzzy_anomaly_score",
        "distance_membership", "detector_activation_membership",
        "min_detector_distance", "mean_detector_distance",
        "detector_distance_std", "min_activation_radius",
        "mean_activation_radius", "activation_radius_std",
        "activated_detector_count", "max_detector_penetration",
        "mean_detector_penetration", "fuzzy_decision_margin"
    ]
    gnn_names = [f"gnn_embedding_{i}" for i in range(GNN_OUT)]
    csa_names = [f"csa_probability_{name}" for name in CLASS_NAMES]
    return list(selected_names) + immune_names + gnn_names + csa_names

def classwise_multiclass_metrics(y_true, y_pred, model_name):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    rows = []
    for c, name in enumerate(CLASS_NAMES):
        yt = (y_true == c).astype(int)
        yp = (y_pred == c).astype(int)
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        rows.append({
            "Model": model_name,
            "Class_ID": c,
            "Class": name,
            "Samples": int((y_true == c).sum()),
            "Accuracy_OVR": (tp + tn) / max(tp + tn + fp + fn, 1),
            "Precision": tp / max(tp + fp, 1),
            "Recall": tp / max(tp + fn, 1),
            "ADR": tp / max(tp + fn, 1),
            "F1": (2 * tp) / max(2 * tp + fp + fn, 1)
        })
    return pd.DataFrame(rows)


## Block 19A: Training and Evaluation


In [ ]:
def train_and_evaluate_outer(
    train_df,
    eval_df,
    seed=SEED,
    make_plot=False
):
    pipeline_start = time.perf_counter()
    train_df = (
        train_df
        .reset_index(
            drop=True
        )
        .copy()
    )

    eval_df = (
        eval_df
        .reset_index(
            drop=True
        )
        .copy()
    )

    # Decision-validation split.
    fit_idx, cal_idx = (
        group_stratified_split_indices(
            train_df,
            y_col="multi_label",
            n_splits=5,
            fold=0
        )
    )

    fit_df = (
        train_df
        .iloc[
            fit_idx
        ]
        .reset_index(
            drop=True
        )
    )

    cal_df = (
        train_df
        .iloc[
            cal_idx
        ]
        .reset_index(
            drop=True
        )
    )

    print(
        "Outer train -> "
        f"fit={len(fit_df)}, "
        f"decision-val={len(cal_df)}, "
        f"eval={len(eval_df)}"
    )

    # HHO and preprocessing
    pre, sel_idx, sel_names, hho_hist = (
        fit_hho_preprocessor(
            fit_df
        )
    )

    X_fit_all = (
        pre.transform(
            fit_df
        )
    )

    X_cal_all = (
        pre.transform(
            cal_df
        )
    )

    X_eval_all = (
        pre.transform(
            eval_df
        )
    )

    X_fit = (
        X_fit_all[
            :,
            sel_idx
        ]
    )

    X_cal = (
        X_cal_all[
            :,
            sel_idx
        ]
    )

    X_eval = (
        X_eval_all[
            :,
            sel_idx
        ]
    )

    y_fit_bin = (
        fit_df[
            "binary_label"
        ].to_numpy()
    )

    y_fit_mc = (
        fit_df[
            "multi_label"
        ].to_numpy()
    )

    y_cal_bin = (
        cal_df[
            "binary_label"
        ].to_numpy()
    )

    y_cal_mc = (
        cal_df[
            "multi_label"
        ].to_numpy()
    )

    meta_fit = (
        raw_graph_meta(
            fit_df
        )
    )

    meta_cal = (
        raw_graph_meta(
            cal_df
        )
    )

    meta_eval = (
        raw_graph_meta(
            eval_df
        )
    )

    categorical_names = set(
        pre.cat_cols
    )

    # WGAN-GP
    (
        X_aug,
        y_aug_bin,
        y_aug_mc,
        meta_aug
    ) = wgan_gp_oversample(
        X_fit,
        y_fit_mc,
        meta_fit,
        sel_names,
        categorical_names,
        forbidden_arrays=[
            X_cal,
            X_eval
        ],
        seed=seed
    )

    wgan_audit = wgan_gp_oversample.last_audit.copy()

    # Out-of-fold upstream features
    (
        Z_nsa_oof,
        Z_gnn_oof,
        Z_csa_oof
    ) = crossfit_upstream(
        X_fit,
        y_fit_bin,
        y_fit_mc,
        meta_fit,
        fit_df[
            "_dup_group"
        ].astype(str).to_numpy(),
        sel_names,
        categorical_names,
        seed=seed
    )

    Z_fit_real = (
        assemble_shared_embedding(
            X_fit,
            Z_nsa_oof,
            Z_gnn_oof,
            Z_csa_oof
        )
    )

    # Final upstream models
    # Tune the MFV-NSA threshold from cross-fitted training scores.
    tau_oof = choose_binary_threshold(
        y_fit_bin,
        Z_nsa_oof[:, 1]
    )

    upstream = (
        fit_final_upstream(
            X_aug,
            y_aug_bin,
            y_aug_mc,
            meta_aug,
            seed=seed
        )
    )
    upstream.nsa.tau = float(tau_oof)
    print(f"Training-OOF MFV-NSA tau*: {upstream.nsa.tau:.3f}")

    (
        Z_nsa_cal,
        Z_gnn_cal,
        Z_csa_cal
    ) = transform_upstream(
        X_cal,
        meta_cal,
        upstream
    )

    (
        Z_nsa_eval,
        Z_gnn_eval,
        Z_csa_eval
    ) = transform_upstream(
        X_eval,
        meta_eval,
        upstream
    )

    Z_cal = (
        assemble_shared_embedding(
            X_cal,
            Z_nsa_cal,
            Z_gnn_cal,
            Z_csa_cal
        )
    )

    Z_eval = (
        assemble_shared_embedding(
            X_eval,
            Z_nsa_eval,
            Z_gnn_eval,
            Z_csa_eval
        )
    )

    # Transform synthetic rows with final upstream models.
    (
        Z_nsa_aug,
        Z_gnn_aug,
        Z_csa_aug
    ) = transform_upstream(
        X_aug,
        meta_aug,
        upstream
    )

    Z_aug_all = (
        assemble_shared_embedding(
            X_aug,
            Z_nsa_aug,
            Z_gnn_aug,
            Z_csa_aug
        )
    )

    # Locate WGAN-generated rows.
    real_hash = row_hashes(
        X_fit
    )

    is_synth = []

    for row in np.round(
        X_aug,
        8
    ):
        h = hashlib.sha256(
            np.ascontiguousarray(
                row,
                dtype=np.float64
            ).tobytes()
        ).hexdigest()

        is_synth.append(
            h not in real_hash
        )

    is_synth = np.asarray(
        is_synth,
        dtype=bool
    )

    Z_train = np.vstack([
        Z_fit_real,
        Z_aug_all[
            is_synth
        ]
    ])

    y_train_bin = np.concatenate([
        y_fit_bin,
        y_aug_bin[
            is_synth
        ]
    ])

    y_train_mc = np.concatenate([
        y_fit_mc,
        y_aug_mc[
            is_synth
        ]
    ])

    print(
        "Shared embedding dimension:",
        Z_train.shape[1]
    )

    print(
        "CNN training rows:",
        len(Z_train),
        "| validation rows:",
        len(Z_cal)
    )

    # CNN-BiLSTM-MHA
    tf.keras.backend.clear_session()

    cnn = build_cnn_bilstm_mha(
        Z_train.shape[1]
    )

    cnn_start = time.perf_counter()

    history = cnn.fit(
        seqify(
            Z_train
        ),
        {
            "binary":
            y_train_bin,
            "multiclass":
            y_train_mc
        },
        validation_data=(
            seqify(
                Z_cal
            ),
            {
                "binary":
                y_cal_bin,
                "multiclass":
                y_cal_mc
            }
        ),
        epochs=CNN_EPOCHS,
        batch_size=CNN_BATCH,
        verbose=1,
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=7,
                restore_best_weights=True
            ),
            ReduceLROnPlateau(
                monitor="val_loss",
                patience=3,
                factor=0.5,
                min_lr=1e-5
            )
        ]
    )

    cnn_train_seconds = time.perf_counter() - cnn_start
    cnn_epochs_ran = len(history.history.get("loss", []))

    (
        cal_bin_p,
        cal_mc_p
    ) = cnn.predict(
        seqify(
            Z_cal
        ),
        verbose=0
    )

    (
        eval_bin_p,
        eval_mc_p
    ) = cnn.predict(
        seqify(
            Z_eval
        ),
        verbose=0
    )

    cal_bin_p = (
        cal_bin_p.ravel()
    )

    eval_bin_p = (
        eval_bin_p.ravel()
    )

    cnn_threshold = choose_binary_threshold(
        y_cal_bin,
        cal_bin_p
    )

    # Decision fusion
    nsa_cal_score = (
        upstream.nsa.score(
            X_cal
        )
    )

    nsa_eval_score = (
        upstream.nsa.score(
            X_eval
        )
    )

    fusion = (
        CalibratedDecisionFusion()
        .fit(
            nsa_cal_score,
            cal_bin_p,
            cal_mc_p,
            Z_csa_cal,
            y_cal_bin,
            y_cal_mc
        )
    )

    (
        pred_bin,
        pred_mc,
        p_fuse,
        q_mc
    ) = fusion.predict(
        nsa_eval_score,
        eval_bin_p,
        eval_mc_p,
        Z_csa_eval
    )

    # Evaluate after prediction.
    y_eval_bin = (
        eval_df[
            "binary_label"
        ].to_numpy()
    )

    y_eval_mc = (
        eval_df[
            "multi_label"
        ].to_numpy()
    )

    bmet = binary_metrics(
        y_eval_bin,
        pred_bin
    )

    mmet = multiclass_metrics(
        y_eval_mc,
        pred_mc
    )

    print(
        "\nBinary metrics"
    )

    display(
        pd.DataFrame([
            bmet
        ]).round(5)
    )

    print(
        "\nMulti-class metrics"
    )

    display(
        pd.DataFrame([
            mmet
        ]).round(5)
    )

    if make_plot:
        plot_fuzzy_anomaly_distribution(
            X_eval,
            y_eval_bin,
            upstream.nsa,
            "UNSW-NB15"
        )


    pipeline_seconds = time.perf_counter() - pipeline_start

    return {
        "binary": bmet,
        "multiclass": mmet,
        "selected_features": sel_names,
        "hho_history": hho_hist,
        "preprocessor": pre,
        "selected_idx": sel_idx,
        "upstream": upstream,
        "cnn": cnn,
        "cnn_history": history,
        "cnn_train_seconds": cnn_train_seconds,
        "cnn_epochs_ran": cnn_epochs_ran,
        "cnn_threshold": cnn_threshold,
        "nsa_tau_oof": upstream.nsa.tau,
        "wgan_audit": wgan_audit,
        "fusion": fusion,
        "pipeline_seconds": pipeline_seconds,

        "fit_df": fit_df,
        "cal_df": cal_df,
        "eval_df": eval_df,

        "X_fit_selected": X_fit,
        "X_cal_selected": X_cal,
        "X_eval_selected": X_eval,
        "X_aug": X_aug,

        "y_fit_bin": y_fit_bin,
        "y_fit_mc": y_fit_mc,
        "y_cal_bin": y_cal_bin,
        "y_cal_mc": y_cal_mc,
        "y_aug_bin": y_aug_bin,
        "y_aug_mc": y_aug_mc,
        "y_train_bin": y_train_bin,
        "y_train_mc": y_train_mc,
        "y_eval_bin": y_eval_bin,
        "y_eval_mc": y_eval_mc,

        "meta_fit": meta_fit,
        "meta_cal": meta_cal,
        "meta_eval": meta_eval,
        "meta_aug": meta_aug,

        "Z_train": Z_train,
        "Z_cal": Z_cal,
        "Z_eval": Z_eval,
        "Z_nsa_eval": Z_nsa_eval,
        "Z_gnn_eval": Z_gnn_eval,
        "Z_csa_eval": Z_csa_eval,

        "cal_bin_prob": cal_bin_p,
        "cal_mc_prob": cal_mc_p,
        "eval_bin_prob": eval_bin_p,
        "eval_mc_prob": eval_mc_p,

        "eval_pred_bin": pred_bin,
        "eval_pred_mc": pred_mc,
        "eval_fuzzy": nsa_eval_score,
        "eval_fusion_prob": p_fuse,
        "eval_fusion_mc_prob": q_mc
    }


## Block 19B: Nested Five-Fold Cross-Validation


In [ ]:
nested_binary_rows = []
nested_multiclass_rows = []

if RUN_NESTED_CV:
    splitter = (
        StratifiedGroupKFold(
            n_splits=OUTER_FOLDS,
            shuffle=True,
            random_state=SEED
        )
    )

    y_dev = (
        dev_df[
            "multi_label"
        ].to_numpy()
    )

    groups_dev = (
        dev_df[
            "_dup_group"
        ]
        .astype(str)
        .to_numpy()
    )

    for fold, (
        tr_idx,
        va_idx
    ) in enumerate(
        splitter.split(
            np.zeros(
                len(dev_df)
            ),
            y_dev,
            groups_dev
        ),
        1
    ):
        print(
            "\n"
            + "=" * 90
        )

        print(
            f"NESTED OUTER FOLD "
            f"{fold}/{OUTER_FOLDS}"
        )

        print(
            "=" * 90
        )

        out = (
            train_and_evaluate_outer(
                dev_df.iloc[
                    tr_idx
                ].reset_index(
                    drop=True
                ),
                dev_df.iloc[
                    va_idx
                ].reset_index(
                    drop=True
                ),
                seed=(
                    SEED
                    + 100 * fold
                ),
                make_plot=False
            )
        )

        nested_binary_rows.append({
            "Fold":
            fold,
            **out[
                "binary"
            ]
        })

        nested_multiclass_rows.append({
            "Fold":
            fold,
            **out[
                "multiclass"
            ]
        })

    nested_binary_df = (
        pd.DataFrame(
            nested_binary_rows
        )
    )

    nested_multiclass_df = (
        pd.DataFrame(
            nested_multiclass_rows
        )
    )

    print(
        "\nNested 5-fold — Binary"
    )

    display(
        nested_binary_df.round(
            5
        )
    )

    print(
        "\nBinary Mean ± SD"
    )

    display(
        pd.DataFrame({
            "Mean":
            nested_binary_df
            .drop(
                columns=[
                    "Fold"
                ]
            )
            .mean(
                numeric_only=True
            ),

            "SD":
            nested_binary_df
            .drop(
                columns=[
                    "Fold"
                ]
            )
            .std(
                numeric_only=True
            )
        }).round(5)
    )

    print(
        "\nNested 5-fold — Multi-class"
    )

    display(
        nested_multiclass_df.round(
            5
        )
    )

    print(
        "\nMulti-class Mean ± SD"
    )

    display(
        pd.DataFrame({
            "Mean":
            nested_multiclass_df
            .drop(
                columns=[
                    "Fold"
                ]
            )
            .mean(
                numeric_only=True
            ),

            "SD":
            nested_multiclass_df
            .drop(
                columns=[
                    "Fold"
                ]
            )
            .std(
                numeric_only=True
            )
        }).round(5)
    )

else:
    print(
        "RUN_NESTED_CV=False: "
        "nested CV skipped."
    )


## Block 20: Final Training and Test Evaluation


In [ ]:
final_output = None
analysis_graphs = None
feature_analysis_df = None

if RUN_FINAL_LOCKED_TEST:
    print("\n" + "=" * 90)
    print("FINAL LOCKED UNSW-NB15 TEST EVALUATION")
    print("=" * 90)

    final_output = train_and_evaluate_outer(
        dev_df,
        test_df,
        seed=SEED + 999,
        make_plot=True
    )

    final_binary_df = pd.DataFrame([final_output["binary"]])
    final_multiclass_df = pd.DataFrame([final_output["multiclass"]])

    print("\nFINAL BINARY CLASSIFICATION")
    display(final_binary_df.round(5))

    print("\nFINAL MULTI-CLASS CLASSIFICATION")
    display(final_multiclass_df.round(5))

    print("\nSelected HHO features:")
    print(final_output["selected_features"])

    print("\nWGAN-GP synthetic-data novelty audit")
    display(final_output["wgan_audit"].round(6))

    print(
        "\nMFV-NSA summary:",
        {
            "self_set_size": len(final_output["upstream"].nsa.self_set),
            "boundary_radius": final_output["upstream"].nsa.r_boundary,
            "detectors": len(final_output["upstream"].nsa.detectors),
            "tau_oof": final_output["nsa_tau_oof"]
        }
    )

    history = final_output["cnn_history"]
    if history is not None:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        if "binary_loss" in history.history:
            axes[0].plot(history.history["binary_loss"], label="Train")
            axes[0].plot(history.history.get("val_binary_loss", []), label="Validation")
        else:
            axes[0].plot(history.history.get("loss", []), label="Train")
            axes[0].plot(history.history.get("val_loss", []), label="Validation")
        axes[0].set_title("Binary Head / Total Loss")
        axes[0].legend()

        if "multiclass_loss" in history.history:
            axes[1].plot(history.history["multiclass_loss"], label="Train")
            axes[1].plot(history.history.get("val_multiclass_loss", []), label="Validation")
        else:
            axes[1].plot(history.history.get("loss", []), label="Train")
            axes[1].plot(history.history.get("val_loss", []), label="Validation")
        axes[1].set_title("Multi-Class Head / Total Loss")
        axes[1].legend()
        plt.suptitle("CNN-BiLSTM-MHA Training Curves", fontweight="bold")
        plt.tight_layout()
        plt.show()

    print("\nGraph analysis")
    analysis_graphs = run_graph_analysis(final_output)

    print("\nFlow-level and packet-level feature analysis")
    feature_analysis_df = run_feature_analysis(final_output)

else:
    print("RUN_FINAL_LOCKED_TEST=False: final locked test skipped.")


In [ ]:
binary_model_outputs = {}
trained_binary_models = {}
binary_training_time_rows = []

def _tune_probability_threshold(y_true, prob):
    best_t, best_f = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 91):
        pred = (np.asarray(prob) >= t).astype(int)
        f = f1_score(y_true, pred, zero_division=0)
        if f > best_f:
            best_f, best_t = f, float(t)
    return best_t

def make_binary_model(name):
    if name == "MLP":
        return MLPClassifier(
            hidden_layer_sizes=(128, 64), activation="relu",
            solver="adam", max_iter=120, early_stopping=True,
            random_state=SEED
        )
    if name == "DT":
        return DecisionTreeClassifier(
            random_state=SEED, class_weight="balanced", min_samples_leaf=2
        )
    if name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=400, random_state=SEED, n_jobs=-1,
            class_weight="balanced"
        )
    if name == "XGBoost":
        return XGBClassifier(
            n_estimators=800, max_depth=8, learning_rate=0.03,
            subsample=0.9, colsample_bytree=0.9,
            objective="binary:logistic", eval_metric="logloss",
            random_state=SEED, n_jobs=-1
        )
    if name == "LightGBM":
        return LGBMClassifier(
            n_estimators=800, learning_rate=0.03, num_leaves=64,
            subsample=0.9, colsample_bytree=0.9,
            random_state=SEED, class_weight="balanced",
            n_jobs=-1, verbose=-1
        )
    if name == "AdaBoost":
        tree = DecisionTreeClassifier(max_depth=2, random_state=SEED)
        try:
            return AdaBoostClassifier(
                estimator=tree, n_estimators=250,
                learning_rate=0.05, random_state=SEED
            )
        except TypeError:
            return AdaBoostClassifier(
                base_estimator=tree, n_estimators=250,
                learning_rate=0.05, random_state=SEED
            )
    if name == "Gradient Boosting":
        return GradientBoostingClassifier(
            n_estimators=250, learning_rate=0.05,
            max_depth=3, random_state=SEED
        )
    if name == "Bagging":
        tree = DecisionTreeClassifier(
            random_state=SEED, class_weight="balanced"
        )
        try:
            return BaggingClassifier(
                estimator=tree, n_estimators=150,
                max_samples=0.8, max_features=0.8,
                random_state=SEED, n_jobs=-1
            )
        except TypeError:
            return BaggingClassifier(
                base_estimator=tree, n_estimators=150,
                max_samples=0.8, max_features=0.8,
                random_state=SEED, n_jobs=-1
            )
    if name == "Stacking":
        return StackingClassifier(
            estimators=[
                ("rf", make_binary_model("Random Forest")),
                ("xgb", make_binary_model("XGBoost")),
                ("lgbm", make_binary_model("LightGBM")),
                ("bag", make_binary_model("Bagging")),
                ("gb", make_binary_model("Gradient Boosting"))
            ],
            final_estimator=LogisticRegression(
                max_iter=1000, class_weight="balanced", random_state=SEED
            ),
            stack_method="predict_proba",
            passthrough=True,
            cv=5,
            n_jobs=-1
        )
    raise ValueError(f"Unknown binary model: {name}")

if final_output is not None:
    y_test_bin = final_output["y_eval_bin"]

    # CNN
    cnn_prob = final_output["eval_bin_prob"]
    cnn_pred = (cnn_prob >= final_output["cnn_threshold"]).astype(int)
    binary_model_outputs["CNN-BiLSTM-MHA"] = {
        "prob": cnn_prob,
        "pred": cnn_pred,
        "metrics": binary_curve_metrics(y_test_bin, cnn_pred, cnn_prob)
    }
    binary_training_time_rows.append({
        "Model": "CNN-BiLSTM-MHA",
        "Training_Time_min": final_output["cnn_train_seconds"] / 60.0,
        "Epochs": final_output["cnn_epochs_ran"],
        "Epoch_Time_sec": final_output["cnn_train_seconds"] / max(final_output["cnn_epochs_ran"], 1)
    })

    # Binary baselines
    if RUN_COMPARATIVE_MODELS:
        model_names = [
            "MLP", "Random Forest", "DT", "XGBoost", "LightGBM",
            "AdaBoost", "Gradient Boosting", "Bagging", "Stacking"
        ]

        X_train = final_output["Z_train"]
        y_train = final_output["y_train_bin"] if "y_train_bin" in final_output else None
        if y_train is None:
            raise KeyError("y_train_bin missing from final_output.")
        X_cal = final_output["Z_cal"]
        y_cal = final_output["y_cal_bin"]
        X_test = final_output["Z_eval"]

        for name in model_names:
            print(f"\nTraining final binary {name}")
            model = make_binary_model(name)
            t0 = time.perf_counter()
            model.fit(X_train, y_train)
            elapsed = time.perf_counter() - t0

            cal_prob = model.predict_proba(X_cal)[:, 1]
            threshold = _tune_probability_threshold(y_cal, cal_prob)
            test_prob = model.predict_proba(X_test)[:, 1]
            test_pred = (test_prob >= threshold).astype(int)

            trained_binary_models[name] = model
            binary_model_outputs[name] = {
                "prob": test_prob,
                "pred": test_pred,
                "threshold": threshold,
                "metrics": binary_curve_metrics(y_test_bin, test_pred, test_prob)
            }

            epochs = getattr(model, "n_iter_", np.nan)
            if isinstance(epochs, np.ndarray):
                epochs = int(np.max(epochs))
            binary_training_time_rows.append({
                "Model": name,
                "Training_Time_min": elapsed / 60.0,
                "Epochs": epochs,
                "Epoch_Time_sec": elapsed / max(float(epochs), 1.0)
                if np.isfinite(epochs) else np.nan
            })

    # MFV-NSA
    nsa_prob = final_output["eval_fuzzy"]
    nsa_pred = (nsa_prob >= final_output["upstream"].nsa.tau).astype(int)
    binary_model_outputs["MFV-NSA"] = {
        "prob": nsa_prob,
        "pred": nsa_pred,
        "metrics": binary_curve_metrics(y_test_bin, nsa_pred, nsa_prob)
    }

    # CSA
    csa_prob = 1.0 - final_output["Z_csa_eval"][:, 0]
    csa_pred = (np.argmax(final_output["Z_csa_eval"], axis=1) != 0).astype(int)
    binary_model_outputs["CSA"] = {
        "prob": csa_prob,
        "pred": csa_pred,
        "metrics": binary_curve_metrics(y_test_bin, csa_pred, csa_prob)
    }

    # Fusion
    fusion_prob = final_output["eval_fusion_prob"]
    fusion_pred = final_output["eval_pred_bin"]
    binary_model_outputs["Decision Fusion"] = {
        "prob": fusion_prob,
        "pred": fusion_pred,
        "metrics": binary_curve_metrics(y_test_bin, fusion_pred, fusion_prob)
    }

    # Hard/soft voting
    vote_names = [
        n for n in [
            "CNN-BiLSTM-MHA", "MLP", "Random Forest", "DT", "XGBoost",
            "LightGBM", "AdaBoost", "Gradient Boosting", "Bagging", "Stacking"
        ] if n in binary_model_outputs
    ]
    if vote_names:
        pred_matrix = np.column_stack([binary_model_outputs[n]["pred"] for n in vote_names])
        prob_matrix = np.column_stack([binary_model_outputs[n]["prob"] for n in vote_names])
        hard_pred = (pred_matrix.mean(axis=1) >= 0.5).astype(int)
        soft_prob = prob_matrix.mean(axis=1)
        soft_pred = (soft_prob >= 0.5).astype(int)

        binary_model_outputs["Hard Voting"] = {
            "prob": soft_prob,
            "pred": hard_pred,
            "metrics": binary_curve_metrics(y_test_bin, hard_pred, soft_prob)
        }
        binary_model_outputs["Soft Voting"] = {
            "prob": soft_prob,
            "pred": soft_pred,
            "metrics": binary_curve_metrics(y_test_bin, soft_pred, soft_prob)
        }

    binary_results_df = pd.DataFrame([
        {
            "Model": name,
            **{k: v for k, v in item["metrics"].items()
               if k not in {"roc_fpr", "roc_tpr", "pr_precision", "pr_recall"}}
        }
        for name, item in binary_model_outputs.items()
    ])
    print("\nFINAL BINARY RESULTS")
    display(binary_results_df.round(5))


## Block 20C: Timing and Five-Fold Results


In [ ]:
multiclass_training_time_rows = []
multiclass_model_outputs = {}
trained_multiclass_models = {}
cv_multiclass_rows = []
cv_multiclass_fold_outputs = {}

def multiclass_average_metrics(y_true, y_pred, model_name):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    labels = list(range(NUM_CLASSES))

    acc = accuracy_score(y_true, y_pred)
    precision_macro = precision_score(
        y_true, y_pred, labels=labels, average="macro", zero_division=0
    )
    recall_macro = recall_score(
        y_true, y_pred, labels=labels, average="macro", zero_division=0
    )
    f1_macro = f1_score(
        y_true, y_pred, labels=labels, average="macro", zero_division=0
    )

    fprs, tnrs, attack_recalls = [], [], []
    for c in labels:
        yt = (y_true == c).astype(int)
        yp = (y_pred == c).astype(int)
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        fprs.append(fp / max(fp + tn, 1))
        tnrs.append(tn / max(tn + fp, 1))
        if c != 0:
            attack_recalls.append(tp / max(tp + fn, 1))

    return {
        "Model": model_name,
        "Accuracy": acc,
        "Precision_macro": precision_macro,
        "Recall_macro": recall_macro,
        "ADR": float(np.mean(attack_recalls)) if attack_recalls else 0.0,
        "FPR": float(np.mean(fprs)),
        "TNR": float(np.mean(tnrs)),
        "F1_macro": f1_macro
    }

def make_multiclass_model(name):
    if name == "MLP":
        return MLPClassifier(
            hidden_layer_sizes=(128, 64), activation="relu",
            solver="adam", max_iter=120, early_stopping=True,
            random_state=SEED
        )
    if name == "DT":
        return DecisionTreeClassifier(
            random_state=SEED, class_weight="balanced", min_samples_leaf=2
        )
    if name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=400, random_state=SEED, n_jobs=-1,
            class_weight="balanced"
        )
    if name == "XGBoost":
        return XGBClassifier(
            n_estimators=800, max_depth=8, learning_rate=0.03,
            subsample=0.9, colsample_bytree=0.9,
            objective="multi:softprob", num_class=NUM_CLASSES,
            eval_metric="mlogloss", random_state=SEED, n_jobs=-1
        )
    if name == "LightGBM":
        return LGBMClassifier(
            n_estimators=800, learning_rate=0.03, num_leaves=64,
            subsample=0.9, colsample_bytree=0.9,
            objective="multiclass", num_class=NUM_CLASSES,
            random_state=SEED, class_weight="balanced",
            n_jobs=-1, verbose=-1
        )
    if name == "AdaBoost":
        tree = DecisionTreeClassifier(max_depth=2, random_state=SEED)
        try:
            return AdaBoostClassifier(
                estimator=tree, n_estimators=250,
                learning_rate=0.05, random_state=SEED
            )
        except TypeError:
            return AdaBoostClassifier(
                base_estimator=tree, n_estimators=250,
                learning_rate=0.05, random_state=SEED
            )
    if name == "Gradient Boosting":
        return GradientBoostingClassifier(
            n_estimators=250, learning_rate=0.05,
            max_depth=3, random_state=SEED
        )
    if name == "Bagging":
        tree = DecisionTreeClassifier(
            random_state=SEED, class_weight="balanced"
        )
        try:
            return BaggingClassifier(
                estimator=tree, n_estimators=150,
                max_samples=0.8, max_features=0.8,
                random_state=SEED, n_jobs=-1
            )
        except TypeError:
            return BaggingClassifier(
                base_estimator=tree, n_estimators=150,
                max_samples=0.8, max_features=0.8,
                random_state=SEED, n_jobs=-1
            )
    if name == "Stacking":
        return StackingClassifier(
            estimators=[
                ("rf", make_multiclass_model("Random Forest")),
                ("xgb", make_multiclass_model("XGBoost")),
                ("lgbm", make_multiclass_model("LightGBM")),
                ("bag", make_multiclass_model("Bagging")),
                ("gb", make_multiclass_model("Gradient Boosting"))
            ],
            final_estimator=LogisticRegression(
                max_iter=1000, class_weight="balanced", random_state=SEED
            ),
            stack_method="predict_proba",
            passthrough=True,
            cv=5,
            n_jobs=-1
        )
    raise ValueError(f"Unknown multi-class model: {name}")

def aligned_proba(model, X):
    raw = model.predict_proba(X)
    out = np.zeros((len(X), NUM_CLASSES), dtype=float)
    for j, cls in enumerate(model.classes_):
        out[:, int(cls)] = raw[:, j]
    out /= out.sum(axis=1, keepdims=True) + 1e-12
    return out

if final_output is not None:
    X_train = final_output["Z_train"]
    y_train_bin = final_output["y_train_bin"]
    y_train_mc = final_output["y_train_mc"]
    X_test = final_output["Z_eval"]
    y_test_mc = final_output["y_eval_mc"]

    # CNN multiclass
    cnn_mc_prob = final_output["eval_mc_prob"]
    cnn_mc_pred = np.argmax(cnn_mc_prob, axis=1)
    multiclass_model_outputs["CNN-BiLSTM-MHA"] = {
        "prob": cnn_mc_prob,
        "pred": cnn_mc_pred
    }
    multiclass_training_time_rows.append({
        "Model": "CNN-BiLSTM-MHA",
        "Training_Time_min": final_output["cnn_train_seconds"] / 60.0,
        "Epochs": final_output["cnn_epochs_ran"],
        "Epoch_Time_sec": final_output["cnn_train_seconds"] / max(final_output["cnn_epochs_ran"], 1)
    })

    base_names = [
        "MLP", "Random Forest", "DT", "XGBoost", "LightGBM",
        "AdaBoost", "Gradient Boosting", "Bagging", "Stacking"
    ]

    if RUN_COMPARATIVE_MODELS:
        for name in base_names:
            print(f"\nTraining final multi-class {name}")
            model = make_multiclass_model(name)
            t0 = time.perf_counter()
            model.fit(X_train, y_train_mc)
            elapsed = time.perf_counter() - t0

            pred = model.predict(X_test)
            prob = aligned_proba(model, X_test)

            trained_multiclass_models[name] = model
            multiclass_model_outputs[name] = {"prob": prob, "pred": pred}

            epochs = getattr(model, "n_iter_", np.nan)
            if isinstance(epochs, np.ndarray):
                epochs = int(np.max(epochs))
            multiclass_training_time_rows.append({
                "Model": name,
                "Training_Time_min": elapsed / 60.0,
                "Epochs": epochs,
                "Epoch_Time_sec": elapsed / max(float(epochs), 1.0)
                if np.isfinite(epochs) else np.nan
            })

    # CSA and fusion
    multiclass_model_outputs["CSA"] = {
        "prob": final_output["Z_csa_eval"],
        "pred": np.argmax(final_output["Z_csa_eval"], axis=1)
    }
    multiclass_model_outputs["Decision Fusion"] = {
        "prob": final_output["eval_fusion_mc_prob"],
        "pred": final_output["eval_pred_mc"]
    }

    vote_names = [
        n for n in ["CNN-BiLSTM-MHA"] + base_names
        if n in multiclass_model_outputs
    ]
    if vote_names:
        pred_stack = np.column_stack([
            multiclass_model_outputs[n]["pred"] for n in vote_names
        ])
        hard_pred = []
        for row in pred_stack:
            vals, counts = np.unique(row, return_counts=True)
            hard_pred.append(vals[np.argmax(counts)])
        hard_pred = np.asarray(hard_pred, dtype=int)

        soft_prob = np.mean([
            multiclass_model_outputs[n]["prob"] for n in vote_names
        ], axis=0)
        soft_pred = np.argmax(soft_prob, axis=1)

        multiclass_model_outputs["Hard Voting"] = {
            "prob": soft_prob, "pred": hard_pred
        }
        multiclass_model_outputs["Soft Voting"] = {
            "prob": soft_prob, "pred": soft_pred
        }

    final_multiclass_comparison_df = pd.DataFrame([
        multiclass_average_metrics(y_test_mc, item["pred"], name)
        for name, item in multiclass_model_outputs.items()
    ])
    print("\nAVERAGE MULTI-CLASS FINAL TEST RESULTS")
    display(final_multiclass_comparison_df.round(5))

    binary_train_time_df = pd.DataFrame(binary_training_time_rows)
    multiclass_train_time_df = pd.DataFrame(multiclass_training_time_rows)

    print("\nBINARY TRAINING TIME")
    display(binary_train_time_df.round(4))
    print("\nMULTI-CLASS TRAINING TIME")
    display(multiclass_train_time_df.round(4))

    # Five-fold multiclass evaluation
    if RUN_COMPARATIVE_CV:
        splitter = StratifiedKFold(
            n_splits=COMPARISON_CV_FOLDS,
            shuffle=True,
            random_state=SEED
        )

        cv_names = [n for n in base_names if n in trained_multiclass_models]
        for name in cv_names:
            fold_rows = []
            print(f"\n{COMPARISON_CV_FOLDS}-Fold Multi-Class CV — {name}")
            for fold, (tri, vai) in enumerate(splitter.split(X_train, y_train_mc), 1):
                model = make_multiclass_model(name)
                t0 = time.perf_counter()
                model.fit(X_train[tri], y_train_mc[tri])
                elapsed = time.perf_counter() - t0
                pred = model.predict(X_train[vai])

                met = multiclass_average_metrics(
                    y_train_mc[vai], pred, name
                )
                met["Fold"] = fold
                met["Train_Time_min"] = elapsed / 60.0
                fold_rows.append(met)

            folds_df = pd.DataFrame(fold_rows)
            cv_multiclass_fold_outputs[name] = folds_df
            numeric = folds_df.drop(columns=["Model", "Fold"]).select_dtypes(include=[np.number])
            row = {"Model": name}
            for col in numeric.columns:
                row[f"{col}_mean"] = numeric[col].mean()
                row[f"{col}_std"] = numeric[col].std()
            cv_multiclass_rows.append(row)

        # CNN cross-validation
        fold_rows = []
        oof_cnn_prob = np.zeros((len(X_train), NUM_CLASSES), dtype=float)
        print(f"\n{COMPARISON_CV_FOLDS}-Fold Multi-Class CV — CNN-BiLSTM-MHA")
        for fold, (tri, vai) in enumerate(splitter.split(X_train, y_train_mc), 1):
            tf.keras.backend.clear_session()
            model = build_cnn_bilstm_mha(X_train.shape[1])
            t0 = time.perf_counter()
            hist = model.fit(
                seqify(X_train[tri]),
                {"binary": y_train_bin[tri], "multiclass": y_train_mc[tri]},
                validation_data=(
                    seqify(X_train[vai]),
                    {"binary": y_train_bin[vai], "multiclass": y_train_mc[vai]}
                ),
                epochs=CNN_EPOCHS,
                batch_size=CNN_BATCH,
                verbose=0,
                callbacks=[
                    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
                    ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, min_lr=1e-5)
                ]
            )
            elapsed = time.perf_counter() - t0
            prob = model.predict(seqify(X_train[vai]), verbose=0)[1]
            pred = np.argmax(prob, axis=1)
            oof_cnn_prob[vai] = prob
            met = multiclass_average_metrics(y_train_mc[vai], pred, "CNN-BiLSTM-MHA")
            met["Fold"] = fold
            met["Train_Time_min"] = elapsed / 60.0
            met["Epochs"] = len(hist.history.get("loss", []))
            met["Epoch_Time_sec"] = elapsed / max(met["Epochs"], 1)
            fold_rows.append(met)

        cnn_folds_df = pd.DataFrame(fold_rows)
        cv_multiclass_fold_outputs["CNN-BiLSTM-MHA"] = cnn_folds_df
        numeric = cnn_folds_df.drop(columns=["Model", "Fold"]).select_dtypes(include=[np.number])
        row = {"Model": "CNN-BiLSTM-MHA"}
        for col in numeric.columns:
            row[f"{col}_mean"] = numeric[col].mean()
            row[f"{col}_std"] = numeric[col].std()
        cv_multiclass_rows.append(row)

        cv_multiclass_summary_df = pd.DataFrame(cv_multiclass_rows)
        print("\n5-FOLD MULTI-CLASS SUMMARY")
        display(cv_multiclass_summary_df.round(5))


## Block 20B: Class-Wise Results and Training Time


In [ ]:
classwise_reports = {}

if final_output is not None:
    y_true_mc = final_output["y_eval_mc"]

    cnn_mc_pred = np.argmax(final_output["eval_mc_prob"], axis=1)
    csa_mc_pred = np.argmax(final_output["Z_csa_eval"], axis=1)
    fusion_mc_pred = final_output["eval_pred_mc"]

    for name, pred in [
        ("CNN-BiLSTM-MHA", cnn_mc_pred),
        ("CSA", csa_mc_pred),
        ("Decision Fusion", fusion_mc_pred)
    ]:
        report = classwise_multiclass_metrics(y_true_mc, pred, name)
        classwise_reports[name] = report
        print(f"\nClass-wise Multi-Class ADR — {name}")
        display(
            report[
                ["Class", "Samples", "Accuracy_OVR", "Precision", "Recall", "ADR", "F1"]
            ].round(4)
        )

    training_time_summary_df = pd.DataFrame(binary_training_time_rows)
    print("\nTraining-Time Summary")
    display(training_time_summary_df.round(4))


## Block 21: ROC and Precision-Recall Curves


In [ ]:
if binary_model_outputs:
    plt.figure(figsize=(10, 7))
    for name, item in binary_model_outputs.items():
        m = item["metrics"]
        plt.plot(
            m["roc_fpr"], m["roc_tpr"],
            linewidth=1.5,
            label=f"{name} (AUC={m['ROC_AUC']:.3f})"
        )
    plt.plot([0, 1], [0, 1], "--", linewidth=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Binary ROC Curves — Final Test", fontweight="bold")
    plt.legend(fontsize=8, loc="lower right")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 7))
    for name, item in binary_model_outputs.items():
        m = item["metrics"]
        plt.plot(
            m["pr_recall"], m["pr_precision"],
            linewidth=1.5,
            label=f"{name} (AUC={m['PR_AUC']:.3f})"
        )
    plt.xlabel("Recall / ADR")
    plt.ylabel("Precision")
    plt.title("Binary Precision-Recall Curves — Final Test", fontweight="bold")
    plt.legend(fontsize=8, loc="lower left")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("Run Block 20 before plotting ROC and Precision-Recall curves.")


## Block 21B: Multi-Class ROC Curves


In [ ]:
if final_output is not None and multiclass_model_outputs:
    y_true = final_output["y_eval_mc"]
    classes = np.arange(NUM_CLASSES)
    y_bin = label_binarize(y_true, classes=classes)

    preferred = [
        "CNN-BiLSTM-MHA", "CSA", "Decision Fusion",
        "XGBoost", "LightGBM", "Stacking"
    ]
    sources = {
        name: multiclass_model_outputs[name]["prob"]
        for name in preferred
        if name in multiclass_model_outputs
    }

    for model_name, y_score in sources.items():
        y_score = np.asarray(y_score, dtype=float)
        if y_score.shape[1] != NUM_CLASSES:
            continue

        fpr = {}
        tpr = {}
        roc_auc = {}

        for i in range(NUM_CLASSES):
            if y_bin[:, i].sum() == 0:
                continue
            fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_score[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])

        fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), y_score.ravel())
        roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

        valid = [i for i in range(NUM_CLASSES) if i in fpr]
        all_fpr = np.unique(np.concatenate([fpr[i] for i in valid]))
        mean_tpr = np.zeros_like(all_fpr)
        for i in valid:
            mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
        mean_tpr /= max(len(valid), 1)
        fpr["macro"] = all_fpr
        tpr["macro"] = mean_tpr
        roc_auc["macro"] = auc(all_fpr, mean_tpr)

        plt.figure(figsize=(10, 7))
        plt.plot(
            fpr["micro"], tpr["micro"], linestyle=":",
            linewidth=2.2, label=f"Micro-average AUC={roc_auc['micro']:.3f}"
        )
        plt.plot(
            fpr["macro"], tpr["macro"], linestyle="--",
            linewidth=2.2, label=f"Macro-average AUC={roc_auc['macro']:.3f}"
        )

        for i in valid:
            plt.plot(
                fpr[i], tpr[i], linewidth=1.2,
                label=f"{CLASS_NAMES[i]} AUC={roc_auc[i]:.3f}"
            )

        plt.plot([0, 1], [0, 1], "k--", linewidth=1)
        plt.xscale("log")
        plt.xlim([1e-4, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"UNSW-NB15 Multi-Class ROC — {model_name}", fontweight="bold")
        plt.grid(True, which="both", linestyle="--", linewidth=0.5)
        plt.legend(loc="lower right", fontsize=8)
        plt.tight_layout()
        plt.show()
else:
    print("Run Blocks 20 and 20C before plotting multi-class ROC curves.")


## Block 22: SHAP and LIME


In [ ]:
if RUN_EXPLAINABILITY and final_output is not None:
    feature_names = shared_feature_names(final_output["selected_features"])
    if len(feature_names) != final_output["Z_train"].shape[1]:
        feature_names = [
            f"shared_feature_{i}"
            for i in range(final_output["Z_train"].shape[1])
        ]

    Z_train_df = pd.DataFrame(final_output["Z_train"], columns=feature_names)
    Z_test_df = pd.DataFrame(final_output["Z_eval"], columns=feature_names)

    # SHAP
    if "XGBoost" in trained_binary_models:
        print("\nSHAP Explainability — XGBoost")
        shap_sample = Z_test_df.sample(
            n=min(500, len(Z_test_df)),
            random_state=SEED
        )
        explainer = shap.TreeExplainer(trained_binary_models["XGBoost"])
        shap_values = explainer.shap_values(shap_sample)
        if isinstance(shap_values, list):
            shap_values = shap_values[-1]
        shap.summary_plot(
            shap_values, shap_sample,
            plot_type="bar", show=True
        )
        shap.summary_plot(
            shap_values, shap_sample,
            show=True
        )
    else:
        print("XGBoost model is unavailable; SHAP skipped.")

    # LIME
    if "Stacking" in trained_binary_models:
        print("\nLIME Explainability — Stacking")
        lime_explainer = LimeTabularExplainer(
            training_data=Z_train_df.to_numpy(),
            feature_names=feature_names,
            class_names=["Normal", "Attack"],
            mode="classification",
            discretize_continuous=True,
            random_state=SEED
        )

        stack_item = binary_model_outputs["Stacking"]
        correct_attack = np.where(
            (final_output["y_eval_bin"] == 1)
            & (stack_item["pred"] == 1)
        )[0]
        sample_index = int(correct_attack[0]) if len(correct_attack) else 0

        lime_result = lime_explainer.explain_instance(
            Z_test_df.iloc[sample_index].to_numpy(),
            trained_binary_models["Stacking"].predict_proba,
            num_features=min(15, len(feature_names))
        )
        display(pd.DataFrame(
            lime_result.as_list(),
            columns=["Feature Rule", "Contribution"]
        ))
        fig = lime_result.as_pyplot_figure()
        plt.title("LIME — Stacking Model", fontweight="bold")
        plt.tight_layout()
        plt.show()
    else:
        print("Stacking model is unavailable; LIME skipped.")
else:
    print("Explainability is disabled or the final model has not been evaluated.")


## Block 24: Summary Visualisations


In [ ]:
if final_output is not None:
    if "binary_results_df" in globals() and not binary_results_df.empty:
        summary_metrics = ["Accuracy", "Precision", "Recall", "F1", "TNR"]
        plot_df = binary_results_df.set_index("Model")[summary_metrics]
        ax = plot_df.plot(kind="bar", figsize=(15, 6))
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("Score")
        ax.set_title("Final Binary Test Performance Summary", fontweight="bold")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()

    hho_hist = final_output["hho_history"]
    if len(hho_hist):
        plt.figure(figsize=(8, 4))
        plt.plot(range(1, len(hho_hist) + 1), hho_hist, marker="o", linewidth=2)
        plt.title("HHO Feature-Selection Convergence", fontweight="bold")
        plt.xlabel("Iteration")
        plt.ylabel("Best Fitness")
        plt.grid(True, alpha=0.25)
        plt.tight_layout()
        plt.show()

    csa_labels = final_output["upstream"].csa.cell_labels
    csa_distribution = pd.Series(csa_labels).value_counts().sort_index()
    csa_distribution.index = [
        CLASS_NAMES[i] if i < len(CLASS_NAMES) else str(i)
        for i in csa_distribution.index
    ]
    csa_distribution.plot(kind="bar", figsize=(9, 4))
    plt.title("CSA Memory Cell Class Distribution", fontweight="bold")
    plt.ylabel("Memory Cells")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    gnn_model = final_output["upstream"].gnn_model
    if getattr(gnn_model, "pretrain_losses_", None):
        plt.figure(figsize=(8, 4))
        plt.plot(gnn_model.pretrain_losses_, linewidth=2)
        plt.title("GNN Self-Supervised Pre-Training Loss", fontweight="bold")
        plt.xlabel("Epoch")
        plt.ylabel("Reconstruction MSE")
        plt.tight_layout()
        plt.show()

    if "binary_train_time_df" in globals() and not binary_train_time_df.empty:
        time_df = binary_train_time_df.dropna(subset=["Training_Time_min"])
        if not time_df.empty:
            plt.figure(figsize=(12, 5))
            plt.bar(time_df["Model"], time_df["Training_Time_min"])
            plt.ylabel("Training Time (min)")
            plt.title("Binary Classifier Training Time", fontweight="bold")
            plt.xticks(rotation=35, ha="right")
            plt.tight_layout()
            plt.show()
else:
    print("Run the final evaluation before summary visualisations.")


## Block 25: Save Outputs


In [ ]:
OUTPUT_DIR = Path(os.environ.get("MFV_OUTPUT_DIR", PROJECT_ROOT / "outputs")).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if "nested_binary_df" in globals():
    nested_binary_df.to_csv(
        os.path.join(OUTPUT_DIR, "nested_binary_results.csv"), index=False
    )
if "nested_multiclass_df" in globals():
    nested_multiclass_df.to_csv(
        os.path.join(OUTPUT_DIR, "nested_multiclass_results.csv"), index=False
    )
if "final_binary_df" in globals():
    final_binary_df.to_csv(
        os.path.join(OUTPUT_DIR, "final_binary_results.csv"), index=False
    )
if "final_multiclass_df" in globals():
    final_multiclass_df.to_csv(
        os.path.join(OUTPUT_DIR, "final_multiclass_results.csv"), index=False
    )
if final_output is not None and "wgan_audit" in final_output:
    final_output["wgan_audit"].to_csv(
        os.path.join(OUTPUT_DIR, "wgan_gp_novelty_audit.csv"), index=False
    )
if "binary_results_df" in globals():
    binary_results_df.to_csv(
        os.path.join(OUTPUT_DIR, "binary_model_comparison.csv"), index=False
    )
if "final_multiclass_comparison_df" in globals():
    final_multiclass_comparison_df.to_csv(
        os.path.join(OUTPUT_DIR, "multiclass_model_comparison.csv"), index=False
    )
if "cv_multiclass_summary_df" in globals():
    cv_multiclass_summary_df.to_csv(
        os.path.join(OUTPUT_DIR, "multiclass_5fold_summary.csv"), index=False
    )
if feature_analysis_df is not None:
    feature_analysis_df.to_csv(
        os.path.join(OUTPUT_DIR, "feature_analysis.csv"), index=False
    )
if analysis_graphs is not None:
    analysis_graphs["stats"].to_csv(
        os.path.join(OUTPUT_DIR, "graph_stats.csv"), index=False
    )

for name, report in classwise_reports.items():
    safe_name = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
    report.to_csv(
        os.path.join(OUTPUT_DIR, f"classwise_{safe_name}.csv"), index=False
    )

print("Outputs saved to:", OUTPUT_DIR)


# Adversarial Attack Evaluation


## Block 19: Clean Reference


In [ ]:
# BLOCK 19 ─ Train Defended MFV-GDN Pipeline  (Clean / No-Attack Reference)
# IMPORTANT:
# to the trained objects stored in `final_output` and freezes them as the clean

import os
import time
import copy
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier

if final_output is None:
    raise RuntimeError(
        "Run the original local/GPU final-training/evaluation blocks first "
        "so that `final_output` contains the defended MFV-GDN pipeline."
    )

ATTACK_SEED = int(os.environ.get("MFV_ATTACK_SEED", SEED))
ATTACK_EVAL_N = int(os.environ.get("MFV_ATTACK_EVAL_N", "3000"))
PROBE_N = int(os.environ.get("MFV_ATTACK_PROBE_N", "300"))
QUERY_BUDGET = int(os.environ.get("MFV_ATTACK_QUERY_BUDGET", "30"))
POISON_EPOCHS = int(os.environ.get("MFV_ATTACK_RETRAIN_EPOCHS", "12"))

attack_rng = np.random.default_rng(ATTACK_SEED)

# Freeze clean reference objects.
DEFENDED_UPSTREAM = final_output["upstream"]
DEFENDED_CNN = final_output["cnn"]
DEFENDED_FUSION = final_output["fusion"]

X_ATTACK_FULL = np.asarray(final_output["X_eval_selected"], dtype=np.float32)
META_ATTACK_FULL = final_output["meta_eval"].reset_index(drop=True).copy()
Y_ATTACK_BIN_FULL = np.asarray(final_output["y_eval_bin"], dtype=int)
Y_ATTACK_MC_FULL = np.asarray(final_output["y_eval_mc"], dtype=int)

def _stratified_attack_indices(y_mc, max_n, seed=ATTACK_SEED):
    """Class-aware evaluation subset without changing the locked test set."""
    y_mc = np.asarray(y_mc, dtype=int)
    if max_n <= 0 or max_n >= len(y_mc):
        return np.arange(len(y_mc), dtype=int)

    rng = np.random.default_rng(seed)
    classes = np.unique(y_mc)
    per_class = max(1, max_n // max(len(classes), 1))
    pieces = []

    for cls in classes:
        idx = np.flatnonzero(y_mc == cls)
        if len(idx):
            take = min(per_class, len(idx))
            pieces.append(rng.choice(idx, size=take, replace=False))

    selected = np.concatenate(pieces) if pieces else np.empty(0, dtype=int)

    # Fill any remaining capacity from rows not already selected.
    if len(selected) < max_n:
        remaining = np.setdiff1d(np.arange(len(y_mc)), selected, assume_unique=False)
        if len(remaining):
            fill = rng.choice(
                remaining,
                size=min(max_n - len(selected), len(remaining)),
                replace=False
            )
            selected = np.concatenate([selected, fill])

    rng.shuffle(selected)
    return selected[:max_n]

ATTACK_IDX = _stratified_attack_indices(
    Y_ATTACK_MC_FULL,
    min(ATTACK_EVAL_N, len(Y_ATTACK_MC_FULL))
)

X_ATTACK = X_ATTACK_FULL[ATTACK_IDX].copy()
META_ATTACK = META_ATTACK_FULL.iloc[ATTACK_IDX].reset_index(drop=True).copy()
Y_ATTACK_BIN = Y_ATTACK_BIN_FULL[ATTACK_IDX].copy()
Y_ATTACK_MC = Y_ATTACK_MC_FULL[ATTACK_IDX].copy()

ATTACK_ROWS = np.flatnonzero(Y_ATTACK_BIN == 1)
NORMAL_ROWS = np.flatnonzero(Y_ATTACK_BIN == 0)

print("Defended MFV-GDN clean pipeline frozen for adversarial evaluation.")
print(f"Attack battlefield: {len(X_ATTACK):,} test flows")
print(f"  Attack rows : {len(ATTACK_ROWS):,}")
print(f"  Normal rows : {len(NORMAL_ROWS):,}")
print(f"  Selected feature dimension : {X_ATTACK.shape[1]}")
print(f"  Shared embedding dimension : {final_output['Z_train'].shape[1]}")
print(f"  Attack seed                 : {ATTACK_SEED}")


## Block 20: Attack Harness


In [ ]:
# BLOCK 20 ─ Adversarial Threat Model + Attack Harness

THREAT_MODEL = {
    "Adversary": "External attacker",
    "Goals": [
        "Evade detection",
        "Corrupt training behavior",
        "Infer model behavior or membership",
        "Manipulate graph structure",
        "Tamper with emitted predictions or logs"
    ],
    "Capabilities": [
        "Inject or modify observable network-flow features",
        "Issue adaptive black-box queries",
        "Manipulate protocol/service relationships if the collection path is exposed",
        "Influence training ingestion in the training-time attack scenarios"
    ],
    "Trusted": [
        "local/GPU execution environment",
        "Validation and locked test labels",
        "Clean model parameters and calibration objects",
        "Original split-first partitioning"
    ],
    "Safeguards": [
        "Split-first partitioning and duplicate auditing",
        "Training-only preprocessing and HHO",
        "Training-only WGAN-GP augmentation",
        "Cross-fitted MFV-NSA/GNN/CSA representations",
        "Validation-only calibration and threshold selection",
        "Calibrated multi-signal decision fusion"
    ]
}

print("═" * 84)
print("MFV-GDN ADVERSARIAL THREAT MODEL")
print("═" * 84)
for section, values in THREAT_MODEL.items():
    print(f"\n{section}:")
    if isinstance(values, (list, tuple)):
        for item in values:
            print("  •", item)
    else:
        print(" ", values)

def _flow_embedding_from_graph(graph, gnn_model):
    """Return flow embeddings from a input PyG graph."""
    graph = graph.to(DEVICE)
    gnn_model.eval()
    with torch.no_grad():
        z = gnn_model.flow_embeddings(graph).detach().cpu().numpy()
    return z.astype(np.float32)

def mfv_gdn_attack_forward(
    X_selected,
    meta,
    *,
    upstream=None,
    cnn_model=None,
    fusion_model=None,
    graph_override=None,
    nsa_override=None,
    csa_override=None
):
    """
    Full MFV-GDN forward pass used by every adversarial scenario.

    Parameters
    ----------
    X_selected : selected/scaled flow features produced by the unchanged base pipeline
    meta       : protocol/service metadata aligned row-by-row with X_selected
    upstream   : optional alternate MFV-NSA/GNN/CSA bundle
    cnn_model  : optional alternate CNN-BiLSTM-MHA model
    fusion_model : optional alternate/frozen calibration/fusion object
    graph_override : optional PyG graph for graph-manipulation tests
    nsa_override   : optional tuple (Z_nsa, fuzzy_score)
    csa_override   : optional NUM_CLASSES-wide CSA probability matrix
    """
    X_selected = np.asarray(X_selected, dtype=np.float32)
    meta = meta.reset_index(drop=True)

    up = DEFENDED_UPSTREAM if upstream is None else upstream
    cnn = DEFENDED_CNN if cnn_model is None else cnn_model
    fusion = DEFENDED_FUSION if fusion_model is None else fusion_model

    # MFV-NSA
    if nsa_override is None:
        Z_nsa = up.nsa.transform(X_selected)
        fuzzy_score = up.nsa.score(X_selected)
    else:
        Z_nsa, fuzzy_score = nsa_override
        Z_nsa = np.asarray(Z_nsa, dtype=np.float32)
        fuzzy_score = np.asarray(fuzzy_score, dtype=float)

    # GCN-GAT
    if graph_override is None:
        Z_gnn = gnn_transform(
            X_selected,
            meta,
            up.gnn_mapper,
            up.gnn_model
        )
    else:
        Z_gnn = _flow_embedding_from_graph(
            graph_override,
            up.gnn_model
        )

    # CSA
    if csa_override is None:
        Z_csa = up.csa.predict_proba(X_selected)
    else:
        Z_csa = np.asarray(csa_override, dtype=np.float32)

    if Z_csa.shape != (len(X_selected), NUM_CLASSES):
        raise ValueError(
            f"CSA probability shape {Z_csa.shape} is incompatible with "
            f"({len(X_selected)}, {NUM_CLASSES})."
        )

    # Shared embedding and CNN
    Z_shared = assemble_shared_embedding(
        X_selected,
        Z_nsa,
        Z_gnn,
        Z_csa
    )

    cnn_bin, cnn_mc = cnn.predict(
        seqify(Z_shared),
        verbose=0
    )
    cnn_bin = np.asarray(cnn_bin).ravel()
    cnn_mc = np.asarray(cnn_mc, dtype=float)

    # Decision fusion
    pred_bin, pred_mc, p_fuse, q_mc = fusion.predict(
        fuzzy_score,
        cnn_bin,
        cnn_mc,
        Z_csa
    )

    return {
        "pred_bin": np.asarray(pred_bin, dtype=int),
        "pred_mc": np.asarray(pred_mc, dtype=int),
        "score": np.asarray(p_fuse, dtype=float),
        "fusion_mc_prob": np.asarray(q_mc, dtype=float),
        "fuzzy": np.asarray(fuzzy_score, dtype=float),
        "cnn_bin": cnn_bin,
        "cnn_mc": cnn_mc,
        "csa_prob": Z_csa,
        "Z_nsa": Z_nsa,
        "Z_gnn": Z_gnn,
        "Z_shared": Z_shared
    }

ATTACK_RESULTS = []

def evaluate_attack(
    category,
    scenario,
    y_true_bin,
    result,
    y_true_mc=None,
    extra=None,
    verbose=True
):
    """Record binary robustness metrics plus optional multi-class/rejection metrics."""
    y_true_bin = np.asarray(y_true_bin, dtype=int)
    pred_bin = np.asarray(result["pred_bin"], dtype=int)
    score = np.asarray(result["score"], dtype=float)

    tn, fp, fn, tp = confusion_matrix(
        y_true_bin,
        pred_bin,
        labels=[0, 1]
    ).ravel()

    row = {
        "Category": category,
        "Scenario": scenario,
        "Accuracy": accuracy_score(y_true_bin, pred_bin),
        "Precision": precision_score(y_true_bin, pred_bin, zero_division=0),
        "ADR": recall_score(y_true_bin, pred_bin, zero_division=0),
        "F1": f1_score(y_true_bin, pred_bin, zero_division=0),
        "FPR": fp / max(fp + tn, 1),
        "TNR": tn / max(tn + fp, 1),
        "ROC_AUC": (
            roc_auc_score(y_true_bin, score)
            if len(np.unique(y_true_bin)) > 1 else np.nan
        )
    }

    if y_true_mc is not None and result.get("pred_mc") is not None:
        y_true_mc = np.asarray(y_true_mc, dtype=int)
        pred_mc = np.asarray(result["pred_mc"], dtype=int)
        row["Unknown_Rate"] = float(np.mean(pred_mc == -1))
        row["MC_Accuracy"] = accuracy_score(y_true_mc, pred_mc)
        row["MC_Macro_F1"] = f1_score(
            y_true_mc,
            pred_mc,
            labels=list(range(NUM_CLASSES)),
            average="macro",
            zero_division=0
        )

    if extra:
        row.update(extra)

    ATTACK_RESULTS.append(row)

    if verbose:
        print(
            f"[{category:>14}] {scenario:<42} "
            f"ADR={row['ADR']:.4f}  FPR={row['FPR']:.4f}  "
            f"F1={row['F1']:.4f}  Acc={row['Accuracy']:.4f}"
        )

    return row

print("\nScoring clean no-attack reference through the complete MFV-GDN pipeline...")
CLEAN_RESULT = mfv_gdn_attack_forward(
    X_ATTACK,
    META_ATTACK
)
CLEAN_ROW = evaluate_attack(
    "Reference",
    "Clean (no attack)",
    Y_ATTACK_BIN,
    CLEAN_RESULT,
    Y_ATTACK_MC
)

CLEAN_ADR = CLEAN_ROW["ADR"]
CLEAN_FPR = CLEAN_ROW["FPR"]
CLEAN_F1 = CLEAN_ROW["F1"]


## Block 21: Training-Time Attacks


In [ ]:
# BLOCK 21 ─ TRAINING-TIME ATTACKS   (separated)
#            (a) Data / label poisoning   (b) Duplicate injection
#            (c) GAN contamination

print("\n" + "═" * 84)
print("TRAINING-TIME ATTACKS")
print("═" * 84)

Z_ATTACK_TRAIN = np.asarray(final_output["Z_train"], dtype=np.float32)
Y_ATTACK_TRAIN_BIN = np.asarray(final_output["y_train_bin"], dtype=int)
Y_ATTACK_TRAIN_MC = np.asarray(final_output["y_train_mc"], dtype=int)
Z_ATTACK_CAL = np.asarray(final_output["Z_cal"], dtype=np.float32)
Y_ATTACK_CAL_BIN = np.asarray(final_output["y_cal_bin"], dtype=int)
Y_ATTACK_CAL_MC = np.asarray(final_output["y_cal_mc"], dtype=int)

def _retrain_attack_cnn(Z_train, y_bin, y_mc, epochs=POISON_EPOCHS):
    """Retrain only the final dual-head classifier for a training-time stress test."""
    tf.keras.backend.clear_session()
    model = build_cnn_bilstm_mha(Z_train.shape[1])
    history = model.fit(
        seqify(Z_train),
        {
            "binary": np.asarray(y_bin, dtype=int),
            "multiclass": np.asarray(y_mc, dtype=int)
        },
        validation_data=(
            seqify(Z_ATTACK_CAL),
            {
                "binary": Y_ATTACK_CAL_BIN,
                "multiclass": Y_ATTACK_CAL_MC
            }
        ),
        epochs=epochs,
        batch_size=CNN_BATCH,
        verbose=0,
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=4,
                restore_best_weights=True
            )
        ]
    )
    return model, len(history.history.get("loss", []))

# (a) Data / label poisoning
print("\n(a) Data / label poisoning")
for frac in (0.05, 0.10, 0.20):
    rng = np.random.default_rng(ATTACK_SEED + int(frac * 1000))
    yb_poison = Y_ATTACK_TRAIN_BIN.copy()
    ym_poison = Y_ATTACK_TRAIN_MC.copy()

    n_flip = max(1, int(frac * len(yb_poison)))
    flip_idx = rng.choice(len(yb_poison), size=n_flip, replace=False)

    yb_poison[flip_idx] = 1 - yb_poison[flip_idx]

    for i in flip_idx:
        if ym_poison[i] == 0:
            ym_poison[i] = int(rng.integers(1, NUM_CLASSES))
        else:
            ym_poison[i] = 0

    poisoned_cnn, epochs_ran = _retrain_attack_cnn(
        Z_ATTACK_TRAIN,
        yb_poison,
        ym_poison
    )

    result = mfv_gdn_attack_forward(
        X_ATTACK,
        META_ATTACK,
        cnn_model=poisoned_cnn
    )

    evaluate_attack(
        "Training-time",
        f"Label poisoning {int(frac * 100)}%",
        Y_ATTACK_BIN,
        result,
        Y_ATTACK_MC,
        extra={
            "Poison_Fraction": frac,
            "Poisoned_Rows": n_flip,
            "Retrain_Epochs": epochs_ran
        }
    )

# (b) Duplicate injection
print("\n(b) Duplicate injection")

rng = np.random.default_rng(ATTACK_SEED + 2201)
attack_train_idx = np.flatnonzero(Y_ATTACK_TRAIN_BIN == 1)

if len(attack_train_idx) == 0:
    print("No attack rows available in training representation; duplicate attack skipped.")
else:
    seed_rows = rng.choice(
        attack_train_idx,
        size=min(400, len(attack_train_idx)),
        replace=False
    )

    DUP_COPIES = 25
    dup_Z = np.repeat(
        Z_ATTACK_TRAIN[seed_rows],
        DUP_COPIES,
        axis=0
    )
    dup_Z += rng.normal(
        0.0,
        1e-3,
        size=dup_Z.shape
    ).astype(np.float32)

    dup_yb = np.ones(len(dup_Z), dtype=int)
    dup_ymc = np.repeat(
        Y_ATTACK_TRAIN_MC[seed_rows],
        DUP_COPIES
    )

    Z_flood = np.vstack([
        Z_ATTACK_TRAIN,
        dup_Z
    ]).astype(np.float32)
    yb_flood = np.concatenate([
        Y_ATTACK_TRAIN_BIN,
        dup_yb
    ])
    ymc_flood = np.concatenate([
        Y_ATTACK_TRAIN_MC,
        dup_ymc
    ])

    print(f"Injected {len(dup_Z):,} near-duplicate attack representations.")

    # Without duplicate filtering.
    cnn_no_audit, ep_no_audit = _retrain_attack_cnn(
        Z_flood,
        yb_flood,
        ymc_flood
    )
    result_no_audit = mfv_gdn_attack_forward(
        X_ATTACK,
        META_ATTACK,
        cnn_model=cnn_no_audit
    )
    evaluate_attack(
        "Training-time",
        "Duplicate injection (no audit)",
        Y_ATTACK_BIN,
        result_no_audit,
        Y_ATTACK_MC,
        extra={
            "Injected_Rows": len(dup_Z),
            "Retrain_Epochs": ep_no_audit
        }
    )

    # Near-duplicate filtering.
    nn = NearestNeighbors(
        n_neighbors=2,
        metric="euclidean",
        n_jobs=-1
    ).fit(Z_flood)

    nn_dist, _ = nn.kneighbors(Z_flood)
    second_dist = nn_dist[:, 1]
    dup_eps = float(np.percentile(second_dist, 1) + 1e-6)
    keep = second_dist > dup_eps

    # Keep all classes represented.
    if keep.sum() < NUM_CLASSES:
        keep[:] = True

    removed = int((~keep).sum())
    print(f"Near-duplicate audit removed {removed:,} rows (eps={dup_eps:.3e}).")

    cnn_audited, ep_audited = _retrain_attack_cnn(
        Z_flood[keep],
        yb_flood[keep],
        ymc_flood[keep]
    )
    result_audited = mfv_gdn_attack_forward(
        X_ATTACK,
        META_ATTACK,
        cnn_model=cnn_audited
    )
    evaluate_attack(
        "Training-time",
        "Duplicate injection (audited)",
        Y_ATTACK_BIN,
        result_audited,
        Y_ATTACK_MC,
        extra={
            "Removed_by_Audit": removed,
            "Retrain_Epochs": ep_audited
        }
    )

# (c) GAN contamination
print("\n(c) WGAN-GP augmentation contamination")

X_aug_clean = np.asarray(final_output["X_aug"], dtype=np.float32)
y_aug_bin_clean = np.asarray(final_output["y_aug_bin"], dtype=int)
y_aug_mc_clean = np.asarray(final_output["y_aug_mc"], dtype=int)

normal_pool_idx = np.flatnonzero(y_aug_bin_clean == 0)

if len(normal_pool_idx) == 0:
    print("No normal rows in the augmentation pool; GAN contamination skipped.")
else:
    rng = np.random.default_rng(ATTACK_SEED + 2301)
    n_contam = max(1, int(0.15 * len(X_aug_clean)))
    picked = rng.choice(
        normal_pool_idx,
        size=n_contam,
        replace=True
    )

    contaminated_X = X_aug_clean[picked].copy()
    contaminated_X += rng.normal(
        0.0,
        0.05,
        size=contaminated_X.shape
    ).astype(np.float32)

    X_aug_contam = np.vstack([
        X_aug_clean,
        contaminated_X
    ]).astype(np.float32)

    y_aug_bin_contam = np.concatenate([
        y_aug_bin_clean,
        np.ones(n_contam, dtype=int)
    ])

    y_aug_mc_contam = np.concatenate([
        y_aug_mc_clean,
        rng.integers(
            1,
            NUM_CLASSES,
            size=n_contam,
            dtype=int
        )
    ])

    print(f"Injected {n_contam:,} contaminated augmentation rows.")

    contaminated_nsa = MFVNSA(
        seed=ATTACK_SEED + 2302
    ).fit_core(
        X_aug_contam,
        y_aug_bin_contam
    )
    contaminated_nsa.tau = float(DEFENDED_UPSTREAM.nsa.tau)

    csa_sub_n = min(6000, len(X_aug_contam))
    csa_sub = rng.choice(
        len(X_aug_contam),
        size=csa_sub_n,
        replace=False
    )

    contaminated_csa = ClonalSelectionClassifier(
        seed=ATTACK_SEED + 2303
    ).fit(
        X_aug_contam[csa_sub],
        y_aug_mc_contam[csa_sub]
    )

    Z_nsa_contam = contaminated_nsa.transform(X_ATTACK)
    fuzzy_contam = contaminated_nsa.score(X_ATTACK)
    csa_prob_contam = contaminated_csa.predict_proba(X_ATTACK)

    result_contam = mfv_gdn_attack_forward(
        X_ATTACK,
        META_ATTACK,
        nsa_override=(
            Z_nsa_contam,
            fuzzy_contam
        ),
        csa_override=csa_prob_contam
    )

    evaluate_attack(
        "Training-time",
        "GAN contamination (augmentation pool)",
        Y_ATTACK_BIN,
        result_contam,
        Y_ATTACK_MC,
        extra={
            "Contaminated_Rows": n_contam
        }
    )

print("\nTraining-time attack suite complete.")


## Block 22: Inference-Time Attacks


In [ ]:
# BLOCK 22 ─ INFERENCE-TIME ATTACKS   (separated)
#            (a) Evasion / perturbation   (b) Unseen attack families (zero-day)
#            (c) Adaptive probing

print("\n" + "═" * 84)
print("INFERENCE-TIME ATTACKS")
print("═" * 84)

# (a) Evasion / perturbation
print("\n(a) Evasion / perturbation — L∞ sweep on malicious test flows")

for eps in (0.10, 0.25, 0.50, 1.00):
    rng = np.random.default_rng(
        ATTACK_SEED + int(100 * eps)
    )

    X_ev = X_ATTACK.copy()

    if len(ATTACK_ROWS):
        perturbation = rng.uniform(
            -eps,
            eps,
            size=(len(ATTACK_ROWS), X_ATTACK.shape[1])
        ).astype(np.float32)

        X_ev[ATTACK_ROWS] = (
            X_ev[ATTACK_ROWS]
            + perturbation
        )

    result = mfv_gdn_attack_forward(
        X_ev,
        META_ATTACK
    )

    evaluate_attack(
        "Inference-time",
        f"Evasion L∞={eps:.2f}",
        Y_ATTACK_BIN,
        result,
        Y_ATTACK_MC,
        extra={"Epsilon": eps}
    )

# FGSM on the CNN input.
print("\nWhite-box FGSM stress test — CNN branch in shared-embedding space")

clean_branch = mfv_gdn_attack_forward(
    X_ATTACK,
    META_ATTACK
)

x_tf = tf.convert_to_tensor(
    seqify(clean_branch["Z_shared"]),
    dtype=tf.float32
)

with tf.GradientTape() as tape:
    tape.watch(x_tf)
    bin_prob_tf, _ = DEFENDED_CNN(
        x_tf,
        training=False
    )
    attack_mask_tf = tf.reshape(
        tf.convert_to_tensor(
            Y_ATTACK_BIN.astype(np.float32)
        ),
        (-1, 1, 1)
    )
    # Move against the attack-probability gradient.
    loss = tf.reduce_sum(
        bin_prob_tf * tf.squeeze(attack_mask_tf, axis=-1)
    )

grad = tape.gradient(
    loss,
    x_tf
)

FGSM_EPS = 0.20
x_adv = x_tf - FGSM_EPS * tf.sign(grad)

cnn_adv_bin, cnn_adv_mc = DEFENDED_CNN.predict(
    x_adv,
    verbose=0
)
cnn_adv_bin = np.asarray(cnn_adv_bin).ravel()
cnn_adv_mc = np.asarray(cnn_adv_mc)

pred_bin, pred_mc, p_fuse, q_mc = DEFENDED_FUSION.predict(
    clean_branch["fuzzy"],
    cnn_adv_bin,
    cnn_adv_mc,
    clean_branch["csa_prob"]
)

fgsm_result = {
    "pred_bin": pred_bin,
    "pred_mc": pred_mc,
    "score": p_fuse,
    "fusion_mc_prob": q_mc
}

evaluate_attack(
    "Inference-time",
    f"FGSM ε={FGSM_EPS:.2f} (CNN branch; full fusion)",
    Y_ATTACK_BIN,
    fgsm_result,
    Y_ATTACK_MC,
    extra={"Epsilon": FGSM_EPS}
)

# (b) Unseen attack families / zero-day stress test
print("\n(b) Unseen attack families — classifier-family holdout stress test")

rare_family_candidates = [
    idx for idx, name in enumerate(CLASS_NAMES)
    if str(name).lower() in {"analysis", "backdoor", "worms"}
]

if not rare_family_candidates:
    rare_family_candidates = [
        cls for cls in np.unique(Y_ATTACK_MC)
        if cls != 0
    ][:3]

for family in rare_family_candidates:
    if np.sum(Y_ATTACK_MC == family) == 0:
        continue

    train_mask = Y_ATTACK_TRAIN_MC != family

    if np.sum(train_mask) < 2 or len(np.unique(Y_ATTACK_TRAIN_MC[train_mask])) < 2:
        continue

    zero_day_cnn, epochs_ran = _retrain_attack_cnn(
        Z_ATTACK_TRAIN[train_mask],
        Y_ATTACK_TRAIN_BIN[train_mask],
        Y_ATTACK_TRAIN_MC[train_mask]
    )

    battlefield = np.flatnonzero(
        (Y_ATTACK_MC == family)
        | (Y_ATTACK_BIN == 0)
    )

    if len(battlefield) == 0:
        continue

    result = mfv_gdn_attack_forward(
        X_ATTACK[battlefield],
        META_ATTACK.iloc[battlefield].reset_index(drop=True),
        cnn_model=zero_day_cnn
    )

    family_name = (
        CLASS_NAMES[family]
        if family < len(CLASS_NAMES)
        else str(family)
    )

    evaluate_attack(
        "Inference-time",
        f"Unseen family {family_name}",
        Y_ATTACK_BIN[battlefield],
        result,
        Y_ATTACK_MC[battlefield],
        extra={
            "Heldout_Family": family_name,
            "Retrain_Epochs": epochs_ran
        }
    )

# (c) Adaptive probing
print(
    "\n(c) Adaptive probing — query-limited black-box greedy evasion "
    f"(N≤{PROBE_N}, budget={QUERY_BUDGET})"
)

if len(ATTACK_ROWS) == 0:
    print("No attack rows available; adaptive probing skipped.")
else:
    rng = np.random.default_rng(
        ATTACK_SEED + 2401
    )

    probe_rows = rng.choice(
        ATTACK_ROWS,
        size=min(PROBE_N, len(ATTACK_ROWS)),
        replace=False
    )

    X_orig = X_ATTACK[probe_rows].copy()
    X_cur = X_orig.copy()
    meta_probe = META_ATTACK.iloc[
        probe_rows
    ].reset_index(drop=True).copy()

    EPS_PROBE = 0.50
    STEP = 0.10

    current = mfv_gdn_attack_forward(
        X_cur,
        meta_probe
    )

    current_score = current["score"].copy()
    current_pred = current["pred_bin"].copy()
    evaded = current_pred == 0
    queries = np.ones(len(X_cur), dtype=int)

    for _ in range(QUERY_BUDGET):
        direction = rng.choice(
            np.array([-1.0, 1.0], dtype=np.float32),
            size=X_cur.shape
        )

        proposal = X_cur + STEP * direction
        proposal = np.clip(
            proposal,
            X_orig - EPS_PROBE,
            X_orig + EPS_PROBE
        ).astype(np.float32)

        trial_result = mfv_gdn_attack_forward(
            proposal,
            meta_probe
        )

        better = (
            (trial_result["score"] < current_score)
            & (~evaded)
        )

        X_cur[better] = proposal[better]
        current_score[better] = trial_result["score"][better]
        current_pred[better] = trial_result["pred_bin"][better]

        queries[~evaded] += 1
        evaded |= current_pred == 0

        if evaded.all():
            break

    final_probe = mfv_gdn_attack_forward(
        X_cur,
        meta_probe
    )

    evasion_rate = float(np.mean(evaded))
    mean_linf = float(
        np.mean(
            np.max(
                np.abs(X_cur - X_orig),
                axis=1
            )
        )
    )

    evaluate_attack(
        "Inference-time",
        "Adaptive probing (black-box)",
        np.ones(len(X_cur), dtype=int),
        final_probe,
        extra={
            "Evasion_Rate": evasion_rate,
            "Mean_Queries": float(np.mean(queries)),
            "Mean_Linf": mean_linf
        }
    )

    print(
        f"Adaptive probing evasion rate={evasion_rate:.4f}, "
        f"mean queries={np.mean(queries):.2f}, "
        f"mean L∞={mean_linf:.4f}"
    )

print("\nInference-time attack suite complete.")


## Block 23: Graph and Output Attacks


In [ ]:
# BLOCK 23 ─ GRAPH / OUTPUT ATTACKS   (separated)
#            (a) Node / edge manipulation   (b) Model extraction / inference
#            (c) Prediction / log tampering

print("\n" + "═" * 84)
print("GRAPH / OUTPUT ATTACKS")
print("═" * 84)

# (a) Node / edge manipulation
print("\n(a) Node / edge manipulation")

clean_graph = build_flow_graph(
    X_ATTACK,
    META_ATTACK,
    DEFENDED_UPSTREAM.gnn_mapper,
    y=None,
    training=False,
    seed=ATTACK_SEED
)

def _tamper_graph(
    graph,
    *,
    rewire_fraction=0.0,
    drop_fraction=0.0,
    node_noise_sigma=0.0,
    seed=ATTACK_SEED
):
    rng = np.random.default_rng(seed)

    edge_index = graph.edge_index.clone()
    flow_src = graph.flow_src.clone()
    flow_dst = graph.flow_dst.clone()
    flow_indices = np.asarray(graph.flow_indices).copy()
    x = graph.x.clone()

    n_nodes = int(x.shape[0])
    n_edges = int(edge_index.shape[1])

    if rewire_fraction > 0 and n_edges > 0:
        k = min(
            n_edges,
            max(1, int(rewire_fraction * n_edges))
        )
        cols = rng.choice(
            n_edges,
            size=k,
            replace=False
        )
        new_dst = rng.integers(
            0,
            n_nodes,
            size=k
        )

        edge_index[1, torch.tensor(cols, dtype=torch.long)] = torch.tensor(
            new_dst,
            dtype=torch.long
        )

    if drop_fraction > 0 and n_edges > 1:
        keep = rng.random(n_edges) > drop_fraction
        if not keep.any():
            keep[rng.integers(0, n_edges)] = True
        edge_index = edge_index[:, torch.tensor(keep, dtype=torch.bool)]

    if node_noise_sigma > 0:
        noise = torch.tensor(
            rng.normal(
                0.0,
                node_noise_sigma,
                size=tuple(x.shape)
            ),
            dtype=x.dtype
        )
        x = x + noise

    tampered = PyGData(
        x=x,
        edge_index=edge_index
    )
    tampered.flow_src = flow_src
    tampered.flow_dst = flow_dst
    tampered.flow_indices = flow_indices
    return tampered

graph_scenarios = [
    (
        "Edge rewire 30%",
        dict(rewire_fraction=0.30)
    ),
    (
        "Edge drop 30%",
        dict(drop_fraction=0.30)
    ),
    (
        "Node-feature noise σ=0.5",
        dict(node_noise_sigma=0.50)
    )
]

for j, (name, kwargs) in enumerate(graph_scenarios):
    tampered_graph = _tamper_graph(
        clean_graph,
        seed=ATTACK_SEED + 2500 + j,
        **kwargs
    )

    result = mfv_gdn_attack_forward(
        X_ATTACK,
        META_ATTACK,
        graph_override=tampered_graph
    )

    evaluate_attack(
        "Graph/output",
        name,
        Y_ATTACK_BIN,
        result,
        Y_ATTACK_MC
    )

# (b) Model extraction / inference
print("\n(b) Model extraction / inference")

oracle = CLEAN_RESULT["pred_bin"]

if len(X_ATTACK) >= 4:
    split_point = max(2, len(X_ATTACK) // 2)
    query_idx = np.arange(split_point)
    holdout_idx = np.arange(split_point, len(X_ATTACK))

    if len(holdout_idx):
        surrogate = RandomForestClassifier(
            n_estimators=200,
            random_state=ATTACK_SEED,
            n_jobs=-1
        )
        surrogate.fit(
            X_ATTACK[query_idx],
            oracle[query_idx]
        )

        surrogate_pred = surrogate.predict(
            X_ATTACK[holdout_idx]
        )

        fidelity = float(
            np.mean(
                surrogate_pred
                == oracle[holdout_idx]
            )
        )

        print(
            f"Surrogate fidelity={fidelity:.4f} "
            f"with query budget={len(query_idx):,}"
        )

        ATTACK_RESULTS.append({
            "Category": "Graph/output",
            "Scenario": "Model extraction (surrogate)",
            "Accuracy": np.nan,
            "Precision": np.nan,
            "ADR": np.nan,
            "F1": np.nan,
            "FPR": np.nan,
            "TNR": np.nan,
            "ROC_AUC": np.nan,
            "Surrogate_Fidelity": fidelity,
            "Query_Budget": len(query_idx)
        })

# Membership inference.
Z_member = np.asarray(
    final_output["Z_train"],
    dtype=np.float32
)
y_member = np.asarray(
    final_output["y_train_bin"],
    dtype=int
)

Z_nonmember = np.asarray(
    final_output["Z_eval"],
    dtype=np.float32
)
y_nonmember = np.asarray(
    final_output["y_eval_bin"],
    dtype=int
)

n_mi = min(
    4000,
    len(Z_member),
    len(Z_nonmember)
)

if n_mi >= 2:
    rng = np.random.default_rng(
        ATTACK_SEED + 2601
    )

    member_idx = rng.choice(
        len(Z_member),
        size=n_mi,
        replace=False
    )
    nonmember_idx = rng.choice(
        len(Z_nonmember),
        size=n_mi,
        replace=False
    )

    member_prob = DEFENDED_CNN.predict(
        seqify(Z_member[member_idx]),
        verbose=0
    )[0].ravel()

    nonmember_prob = DEFENDED_CNN.predict(
        seqify(Z_nonmember[nonmember_idx]),
        verbose=0
    )[0].ravel()

    member_conf = np.where(
        y_member[member_idx] == 1,
        member_prob,
        1.0 - member_prob
    )

    nonmember_conf = np.where(
        y_nonmember[nonmember_idx] == 1,
        nonmember_prob,
        1.0 - nonmember_prob
    )

    mi_labels = np.concatenate([
        np.ones(n_mi),
        np.zeros(n_mi)
    ])
    mi_scores = np.concatenate([
        member_conf,
        nonmember_conf
    ])

    mi_auc = float(
        roc_auc_score(
            mi_labels,
            mi_scores
        )
    )
    mi_advantage = abs(
        2.0 * mi_auc - 1.0
    )

    print(
        f"Membership-inference AUC={mi_auc:.4f}, "
        f"advantage={mi_advantage:.4f}"
    )

    ATTACK_RESULTS.append({
        "Category": "Graph/output",
        "Scenario": "Membership inference",
        "Accuracy": np.nan,
        "Precision": np.nan,
        "ADR": np.nan,
        "F1": np.nan,
        "FPR": np.nan,
        "TNR": np.nan,
        "ROC_AUC": mi_auc,
        "MI_Advantage": mi_advantage
    })

# (c) Prediction / log tampering
print("\n(c) Prediction / log tampering")

base = CLEAN_RESULT
base_bin = base["pred_bin"].copy()

def _evaluate_branch_tamper(
    label,
    *,
    fuzzy_score,
    cnn_bin,
    cnn_mc,
    csa_prob
):
    pred_bin, pred_mc, p_fuse, q_mc = DEFENDED_FUSION.predict(
        fuzzy_score,
        cnn_bin,
        cnn_mc,
        csa_prob
    )

    result = {
        "pred_bin": pred_bin,
        "pred_mc": pred_mc,
        "score": p_fuse,
        "fusion_mc_prob": q_mc
    }

    flip_rate = float(
        np.mean(
            np.asarray(pred_bin)
            != base_bin
        )
    )

    evaluate_attack(
        "Graph/output",
        label,
        Y_ATTACK_BIN,
        result,
        Y_ATTACK_MC,
        extra={
            "Decision_Flip_Rate": flip_rate
        }
    )

# Tamper CNN scores.
cnn_bin_tampered = base["cnn_bin"].copy()
cnn_mc_tampered = base["cnn_mc"].copy()

cnn_bin_tampered[ATTACK_ROWS] = 0.0
cnn_mc_tampered[ATTACK_ROWS] = 0.0
cnn_mc_tampered[ATTACK_ROWS, 0] = 1.0

_evaluate_branch_tamper(
    "Tamper CNN output",
    fuzzy_score=base["fuzzy"],
    cnn_bin=cnn_bin_tampered,
    cnn_mc=cnn_mc_tampered,
    csa_prob=base["csa_prob"]
)

# Tamper MFV-NSA scores.
fuzzy_tampered = base["fuzzy"].copy()
fuzzy_tampered[ATTACK_ROWS] = 0.0

_evaluate_branch_tamper(
    "Tamper MFV-NSA output",
    fuzzy_score=fuzzy_tampered,
    cnn_bin=base["cnn_bin"],
    cnn_mc=base["cnn_mc"],
    csa_prob=base["csa_prob"]
)

# Tamper CSA probabilities.
csa_tampered = base["csa_prob"].copy()
csa_tampered[ATTACK_ROWS] = 0.0
csa_tampered[ATTACK_ROWS, 0] = 1.0

_evaluate_branch_tamper(
    "Tamper CSA output",
    fuzzy_score=base["fuzzy"],
    cnn_bin=base["cnn_bin"],
    cnn_mc=base["cnn_mc"],
    csa_prob=csa_tampered
)

print("\nGraph / output attack suite complete.")


## Block 24: Attack Results and Export


In [ ]:
# BLOCK 24 ─ Consolidated Attack Results · Robustness Figures · LaTeX / CSV Export

ATTACK_OUT_DIR = Path(
    "./"
    "MFV_GDN_UNSW_NB15_outputs/adversarial_attack_results"
)
ATTACK_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

attack_results_df = pd.DataFrame(
    ATTACK_RESULTS
)

if attack_results_df.empty:
    raise RuntimeError(
        "No adversarial results are available. Run Blocks 19-23 first."
    )

attack_results_df["ADR_Retention"] = (
    attack_results_df["ADR"]
    / max(CLEAN_ADR, 1e-12)
)
attack_results_df["FPR_Delta"] = (
    attack_results_df["FPR"]
    - CLEAN_FPR
)

display_columns = [
    "Category",
    "Scenario",
    "ADR",
    "FPR",
    "F1",
    "Accuracy",
    "ADR_Retention",
    "FPR_Delta"
]

print("\n" + "═" * 100)
print("MFV-GDN — CONSOLIDATED ADVERSARIAL RESULTS")
print("═" * 100)

display(
    attack_results_df[
        [c for c in display_columns if c in attack_results_df.columns]
    ].round(5)
)

scored_attacks = attack_results_df.dropna(
    subset=["ADR"]
).copy()
scored_attacks = scored_attacks[
    scored_attacks["Category"] != "Reference"
]

if len(scored_attacks):
    attack_category_summary_df = (
        scored_attacks
        .groupby(
            "Category",
            as_index=False
        )[
            [
                "ADR",
                "FPR",
                "F1",
                "Accuracy",
                "ADR_Retention",
                "FPR_Delta"
            ]
        ]
        .mean()
    )

    print("\nMean robustness by attack category")
    display(
        attack_category_summary_df.round(5)
    )
else:
    attack_category_summary_df = pd.DataFrame()

# ADR plot
if len(scored_attacks):
    fig, ax = plt.subplots(
        figsize=(14, 6)
    )
    ax.bar(
        np.arange(len(scored_attacks)),
        scored_attacks["ADR"].to_numpy()
    )
    ax.axhline(
        CLEAN_ADR,
        linestyle="--",
        linewidth=1.5,
        label=f"Clean ADR = {CLEAN_ADR:.3f}"
    )
    ax.set_xticks(
        np.arange(len(scored_attacks))
    )
    ax.set_xticklabels(
        scored_attacks["Scenario"].to_numpy(),
        rotation=55,
        ha="right",
        fontsize=8
    )
    ax.set_ylabel(
        "Attack Detection Rate (ADR)"
    )
    ax.set_title(
        "MFV-GDN Detection Under Adversarial Attack",
        fontweight="bold"
    )
    ax.set_ylim(
        0.0,
        1.02
    )
    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.30
    )
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        ATTACK_OUT_DIR / "adr_under_attack.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.savefig(
        ATTACK_OUT_DIR / "adr_under_attack.pdf",
        bbox_inches="tight"
    )
    plt.show()

    # FPR plot
    fig, ax = plt.subplots(
        figsize=(14, 6)
    )
    ax.bar(
        np.arange(len(scored_attacks)),
        scored_attacks["FPR"].to_numpy()
    )
    ax.axhline(
        CLEAN_FPR,
        linestyle="--",
        linewidth=1.5,
        label=f"Clean FPR = {CLEAN_FPR:.3f}"
    )
    ax.set_xticks(
        np.arange(len(scored_attacks))
    )
    ax.set_xticklabels(
        scored_attacks["Scenario"].to_numpy(),
        rotation=55,
        ha="right",
        fontsize=8
    )
    ax.set_ylabel(
        "False Positive Rate (FPR)"
    )
    ax.set_title(
        "MFV-GDN False Positives Under Adversarial Attack",
        fontweight="bold"
    )
    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.30
    )
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        ATTACK_OUT_DIR / "fpr_under_attack.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.savefig(
        ATTACK_OUT_DIR / "fpr_under_attack.pdf",
        bbox_inches="tight"
    )
    plt.show()

def _latex_escape(value):
    text = str(value)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}"
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

def adversarial_results_to_latex(df):
    cols = [
        "Category",
        "Scenario",
        "ADR",
        "FPR",
        "F1",
        "Accuracy",
        "ADR_Retention"
    ]

    rows = [
        r"\begin{table*}[t]",
        r"\centering",
        r"\caption{MFV-GDN performance under separated adversarial attack scenarios on UNSW-NB15.}",
        r"\label{tab:mfvgdn_adversarial_unsw}",
        r"\begin{tabular}{llrrrrr}",
        r"\toprule",
        r"Category & Scenario & ADR & FPR & F1 & Acc. & ADR ret. \\",
        r"\midrule"
    ]

    for _, row in df[cols].iterrows():
        numeric = []
        for col in [
            "ADR",
            "FPR",
            "F1",
            "Accuracy",
            "ADR_Retention"
        ]:
            numeric.append(
                "--"
                if pd.isna(row[col])
                else f"{float(row[col]):.3f}"
            )

        rows.append(
            f"{_latex_escape(row['Category'])} & "
            f"{_latex_escape(row['Scenario'])} & "
            + " & ".join(numeric)
            + r" \\"
        )

    rows.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table*}"
    ])

    return "\n".join(rows)

latex_table = adversarial_results_to_latex(
    attack_results_df
)

attack_results_df.to_csv(
    ATTACK_OUT_DIR / "attack_results.csv",
    index=False
)

if not attack_category_summary_df.empty:
    attack_category_summary_df.to_csv(
        ATTACK_OUT_DIR / "attack_category_summary.csv",
        index=False
    )

with open(
    ATTACK_OUT_DIR / "attack_results_table.tex",
    "w",
    encoding="utf-8"
) as f:
    f.write(
        latex_table
    )

print("\nAdversarial outputs saved to:")
print(ATTACK_OUT_DIR)
print("  • attack_results.csv")
print("  • attack_category_summary.csv (when scored attacks are available)")
print("  • attack_results_table.tex")
print("  • adr_under_attack.png / .pdf")
print("  • fpr_under_attack.png / .pdf")
